In [1]:
from google.colab import drive
drive.mount('/content/drive')

KB = '/content/drive/MyDrive/sanad-ai-readiness/kb'
%cd $KB
!mkdir -p raw processed/text index
!pip install -q pdfplumber pypdf
!ls -la

Mounted at /content/drive
/content/drive/MyDrive/sanad-ai-readiness/kb
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 106.1 MB/s eta 0:00:00
total 55
-rw------- 1 root root 27224 Jul 30 11:41  fetch.py
drwx------ 2 root root  4096 Jul 30 11:45  index
-rw------- 1 root root  7996 Jul 30 11:43 'manifest.csv .csv'
-rw------- 1 root root  6244 Jul 29 15:27  manifest.schema.json
drwx------ 3 root root  4096 Jul 30 11:45  processed
drwx------ 2 root root  4096 Jul 30 11:45  raw
-rw------- 1 root root   551 Jul 30 11:44 

In [2]:
import os, csv
os.rename('manifest.csv .csv', 'manifest.csv')

with open('manifest.csv', encoding='utf-8-sig') as f:
    hdr = next(csv.reader(f))
print(f'الأعمدة: {len(hdr)}  (المطلوب 26)')
print(hdr)

expected = {'doc_id','tier','title_en','title_ar','issuing_body','country','doc_type',
 'status','sector','year','version','url','url_accessed','http_status','format',
 'language','is_official_translation','sha256','bytes','page_count','itu_factors',
 'itu_dimensions','reference_pair','license_note','local_path','notes'}
missing = expected - set(hdr)
print('\nناقص:', missing if missing else 'لا شيء ✓')

الأعمدة: 26  (المطلوب 26)
['doc_id', 'tier', 'title_en', 'title_ar', 'issuing_body', 'country', 'doc_type', 'status', 'sector', 'year', 'version', 'url', 'url_accessed', 'http_status', 'format', 'language', 'is_official_translation', 'sha256', 'bytes', 'page_count', 'itu_factors', 'itu_dimensions', 'reference_pair', 'license_note', 'local_path', 'notes']

ناقص: لا شيء ✓


In [3]:
!python fetch.py validate --tier core

Validating 8 records with jsonschema

  line 9  USA-NIST-SP80053-2020
      - 'json' is not one of ['pdf', 'html', 'docx', 'xlsx', 'txt', 'unknown']

FAIL: 1 invalid record(s).


In [4]:
import json

with open('manifest.schema.json', encoding='utf-8') as f:
    s = json.load(f)

s['properties']['format']['enum'] = ['pdf', 'html', 'json', 'docx', 'xlsx', 'txt', 'unknown']

with open('manifest.schema.json', 'w', encoding='utf-8') as f:
    json.dump(s, f, indent=2, ensure_ascii=False)
    f.write('\n')

print('✓', s['properties']['format']['enum'])

✓ ['pdf', 'html', 'json', 'docx', 'xlsx', 'txt', 'unknown']


In [5]:
!python fetch.py validate --tier core

Validating 8 records with jsonschema


PASS: 0 invalid record(s).


In [6]:
!python fetch.py fetch --tier core

Fetching 8 document(s)

  GET     SAU-NCA-ECC-2024
      saved SAU-NCA-ECC-2024.pdf  1,231,347 bytes  sha256 146505199e5374d9...
  GET     SAU-SAMA-CSF-2017
invalid pdf header: b'<!DOC'
EOF marker not found
      saved SAU-SAMA-CSF-2017.pdf  32,400 bytes  sha256 c7d4e1083cfbed48...
  GET     SAU-SDAIA-AIADOPT-2025
      saved SAU-SDAIA-AIADOPT-2025.pdf  2,844,472 bytes  sha256 8f225e863f5fe20b...
  GET     SAU-SDAIA-AIETHICS-2025
      saved SAU-SDAIA-AIETHICS-2025.pdf  1,866,649 bytes  sha256 b1b923d44cb39118...
  GET     SAU-SDAIA-GENAIGOV-2025
      saved SAU-SDAIA-GENAIGOV-2025.pdf  4,176,935 bytes  sha256 fb2427075540afee...
  GET     SAU-SDAIA-PDPL-2023
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 36 0 (offset 0)
Ignoring wrong pointing object 57 0 (offset 0)
      saved SAU-SDAIA-PDPL-2023.pdf  3,000,276 bytes  sha256 f832f3024543ddf3...
  GET     USA-NIST-AIRMF-2023
      saved USA-NIST-AIRMF-2023.pdf  1,946,127 bytes  sha256 7576edb531d98488...


In [7]:
!python fetch.py verify

Verifying held copies against recorded hashes

  ALTERED  SAU-SAMA-CSF-2017  local file no longer matches manifest

altered 1 | missing 0 | upstream revised 0


In [8]:
import pathlib
p = pathlib.Path('fetch.py'); s = p.read_text(encoding='utf-8')

old = '''    rows, fieldnames = load_manifest()
    print("Verifying held copies against recorded hashes\\n")'''

new = '''    rows, fieldnames = load_manifest()

    if getattr(args, "rehash", False):
        print("Re-hashing held copies (on-disk file treated as authoritative)\\n")
        done = 0
        for row in rows:
            lp = row.get("local_path")
            if not lp:
                continue
            path = KB / lp
            if not path.exists():
                print(f"  MISSING  {row.get('doc_id')}  ({lp})")
                continue
            data = path.read_bytes()
            digest = _sha256(data)
            prior = row.get("sha256")
            row["sha256"] = digest
            row["bytes"] = len(data)
            if row.get("format") == "pdf":
                row["page_count"] = _page_count(path)
            state = "unchanged" if prior == digest else ("updated" if prior else "new")
            print(f"  {state:<9} {row.get('doc_id'):<26} "
                  f"{len(data):>12,} bytes  {digest[:16]}...  "
                  f"p{row.get('page_count') or '-'}")
            done += 1
        save_manifest(rows, fieldnames)
        print(f"\\nre-hashed {done} record(s); manifest updated")
        return 0

    print("Verifying held copies against recorded hashes\\n")'''

assert old in s, 'لم يُعثر على الموضع'
s = s.replace(old, new).replace(
    '''    p.add_argument("--upstream", action="store_true", help="also re-fetch and compare")''',
    '''    p.add_argument("--upstream", action="store_true", help="also re-fetch and compare")
    p.add_argument("--rehash", action="store_true", help="adopt on-disk files as authoritative")''')
p.write_text(s, encoding='utf-8')
print('✓ تم الترقيع')

✓ تم الترقيع


In [9]:
!python fetch.py verify --rehash

Re-hashing held copies (on-disk file treated as authoritative)

  unchanged SAU-NCA-ECC-2024              1,231,347 bytes  146505199e5374d9...  p56
  updated   SAU-SAMA-CSF-2017             1,557,571 bytes  ce68ce5a31635094...  p56
  unchanged SAU-SDAIA-AIADOPT-2025        2,844,472 bytes  8f225e863f5fe20b...  p48
  unchanged SAU-SDAIA-AIETHICS-2025       1,866,649 bytes  b1b923d44cb39118...  p50
  unchanged SAU-SDAIA-GENAIGOV-2025       4,176,935 bytes  fb2427075540afee...  p30
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 36 0 (offset 0)
Ignoring wrong pointing object 57 0 (offset 0)
  unchanged SAU-SDAIA-PDPL-2023           3,000,276 bytes  f832f3024543ddf3...  p16
  unchanged USA-NIST-AIRMF-2023           1,946,127 bytes  7576edb531d98488...  p48
  unchanged USA-NIST-SP80053-2020        10,442,037 bytes  01f37cf90ea99d92...  p-

re-hashed 8 record(s); manifest updated


In [10]:
!python fetch.py extract --tier core

  OK    SAU-NCA-ECC-2024  56 page(s), 90,556 chars
  OK    SAU-SAMA-CSF-2017  56 page(s), 127,030 chars
  OK    SAU-SDAIA-AIADOPT-2025  48 page(s), 91,208 chars
  OK    SAU-SDAIA-AIETHICS-2025  50 page(s), 103,305 chars
  OK    SAU-SDAIA-GENAIGOV-2025  30 page(s), 37,442 chars
  OK    SAU-SDAIA-PDPL-2023  16 page(s), 33,189 chars
  OK    USA-NIST-AIRMF-2023  48 page(s), 101,943 chars
  SKIP  USA-NIST-SP80053-2020  (format 'txt' not handled)

extracted 7 | empty/scanned 0


In [11]:
import os, csv, json, pathlib

if pathlib.Path('raw/USA-NIST-SP80053-2020.txt').exists():
    os.rename('raw/USA-NIST-SP80053-2020.txt', 'raw/USA-NIST-SP80053-2020.json')

rows = list(csv.DictReader(open('manifest.csv', encoding='utf-8-sig')))
fn = list(rows[0].keys())
for r in rows:
    if r['doc_id'] == 'USA-NIST-SP80053-2020':
        r['format'] = 'json'
        r['local_path'] = 'raw/USA-NIST-SP80053-2020.json'
        r['page_count'] = '0'
with open('manifest.csv', 'w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fn, lineterminator='\n')
    w.writeheader(); w.writerows(rows)

cat = json.load(open('raw/USA-NIST-SP80053-2020.json', encoding='utf-8'))
meta = cat['catalog']['metadata']
print('العنوان :', meta['title'])
print('النسخة  :', meta.get('version'))
print('العائلات:', len(cat['catalog']['groups']))

العنوان : Electronic (OSCAL) Version of NIST SP 800-53 Rev 5.2.0 Controls and SP 800-53A Rev 5.2.0 Assessment Procedures
النسخة  : 5.2.0
العائلات: 20


In [12]:
!python fetch.py chunk --tier core
!python fetch.py report

  SAU-NCA-ECC-2024: 110 chunk(s)
  SAU-SAMA-CSF-2017: 143 chunk(s)
  SAU-SDAIA-AIADOPT-2025: 107 chunk(s)
  SAU-SDAIA-AIETHICS-2025: 120 chunk(s)
  SAU-SDAIA-GENAIGOV-2025: 48 chunk(s)
  SAU-SDAIA-PDPL-2023: 39 chunk(s)
  USA-NIST-AIRMF-2023: 116 chunk(s)

wrote 683 chunks -> kb/processed/chunks.jsonl
KNOWLEDGE BASE COVERAGE
CORE PROGRESS       : [########] 8/8 collected

policy documents    : 18 in manifest, 8 held
pages held          : 304
method references   : 3 (excluded from KB size)

By legal status
  binding                   10
  advisory                   8

By sector
  cross_sector               9
  general                    5
  finance                    4

By document type
  framework                  6
  controls                   2
  guidelines                 2
  implementing_regulation    2
  law                        1
  standard                   1
  regulation                 1
  policy                     1
  report                     1
  strategy                

In [13]:
!python fetch.py verify

Verifying held copies against recorded hashes


altered 0 | missing 0 | upstream revised 0


In [14]:
!pip install -q sentence-transformers
!python index.py build

Indexing 683 chunks
  bm25 : 7,171 unique terms, avg 86 tokens/chunk
modules.json: 100% 349/349 [00:00<00:00, 1.25MB/s]
config_sentence_transformers.json: 100% 124/124 [00:00<00:00, 239kB/s]
README.md: 100% 94.8k/94.8k [00:00<00:00, 81.8MB/s]
sentence_bert_config.json: 100% 52.0/52.0 [00:00<00:00, 251kB/s]
config.json: 100% 743/743 [00:00<00:00, 3.43MB/s]

model.safetensors: downloading bytes:  60% 79.6M/133M [00:01<00:00, 67.6MB/s, 6.98MB/s  ]
model.safetensors: reconstructing file:  50% 67.0M/133M [00:02<00:02, 32.9MB/s]
model.safetensors: downloading bytes: 100% 86.7M/86.7M [00:02<00:00, 36.6MB/s, 7.97MB/s  ]
model.safetensors: reconstructing file: 100% 133M/133M [00:02<00:00, 56.4MB/s, 12.5MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 19317.41it/s]
tokenizer_config.json: 100% 366/366 [00:00<00:00, 967kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 22.1MB/s]
tokenizer.json: 100% 711k/711k [00:00<00:00, 65.3MB/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 281kB/s]
config

In [1]:
!python index.py stats
print("="*70)
!python index.py --no-dense query "how often must the cyber security committee meet" -k 3
print("="*70)
!python index.py query "how often must the cyber security committee meet" -k 3

python3: can't open file '/content/index.py': [Errno 2] No such file or directory
python3: can't open file '/content/index.py': [Errno 2] No such file or directory
python3: can't open file '/content/index.py': [Errno 2] No such file or directory


In [2]:
%cd /content/drive/MyDrive/sanad-ai-readiness/kb
!pwd

[Errno 2] No such file or directory: '/content/drive/MyDrive/sanad-ai-readiness/kb'
/content
/content


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
KB = '/content/drive/MyDrive/sanad-ai-readiness/kb'
!ls -la {KB} && echo "--- index ---" && ls -la {KB}/index

total 70
-rw------- 1 root root 28412 Jul 30 11:59 fetch.py
drwx------ 2 root root  4096 Jul 30 11:45 index
-rw------- 1 root root 14246 Jul 30 12:14 index.py
-rw------- 1 root root  8905 Jul 30 12:03 manifest.csv
-rw------- 1 root root  6260 Jul 30 11:50 manifest.schema.json
drwx------ 2 root root  4096 Jul 30 11:45 processed
drwx------ 2 root root  4096 Jul 30 11:45 raw
--- index ---
total 1026
-rw------- 1 root root     102 Jul 30 12:20 dense_meta.json
-rw------- 1 root root 1049216 Jul 30 12:20 dense.npy


In [5]:
!pip install -q sentence-transformers pdfplumber pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 121.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 114.0 MB/s eta 0:00:00


In [6]:
!cd {KB} && python index.py stats
print("="*70)
!cd {KB} && python index.py --no-dense query "how often must the cyber security committee meet" -k 3
print("="*70)
!cd {KB} && python index.py query "how often must the cyber security committee meet" -k 3

chunks   : 683

by document
  SAU-SAMA-CSF-2017             143
  SAU-SDAIA-AIETHICS-2025       120
  USA-NIST-AIRMF-2023           116
  SAU-NCA-ECC-2024              110
  SAU-SDAIA-AIADOPT-2025        107
  SAU-SDAIA-GENAIGOV-2025        48
  SAU-SDAIA-PDPL-2023            39

by legal status
  advisory                      391
  binding                       292

chunk chars  min 25  mean 933  max 1199
chunk tokens min 3  mean 86  max 148

control-like identifiers found in text: 7 distinct
  CC-1(4), CC-2(1), AT-53(1), AT-24(1), AG-23(1), CS-34(1), ID-20(1)
Indexing 683 chunks
  bm25 : 7,171 unique terms, avg 86 tokens/chunk

Q: how often must the cyber security committee meet

[1] SAU-SAMA-CSF-2017::p43::c0107   (bm25)
    Cyber Security Framework — SAMA [binding]  p43
    Appendix D - How to request a Waiver from the Framework Below the illustration of the process for requesting a waiver from the Framework.  Detail description about the reasons that the bank could not meet the r

In [7]:
import json
for l in open(f'{KB}/processed/chunks.jsonl', encoding='utf-8'):
    c = json.loads(l)
    if c['chunk_id'] == 'SAU-SAMA-CSF-2017::p13::c0034':
        print(c['text'])
        print('\n>>> quarterly موجودة؟', 'quarterly' in c['text'].lower())

ber security within the Member Organization.
Control considerations
1. A cyber security committee should be established and be mandated by the board.
2. The cyber security committee should be headed by an independent senior manager from a control
function.
3. The following positions should be represented in the cyber security committee:
a. senior managers from all relevant departments (e.g., COO, CIO, compliance officer, heads of
relevant business departments);
b. Chief information security officer (CISO);
c. Internal audit may attend as an “observer.
4. A cyber security committee charter should be developed, approved and reflect:
a. committee objectives;
b. roles and responsibilities;
c. minimum number of meeting participants;
d. meeting frequency (minimum on quarterly basis).
5. A cyber security function should be established.
6. The cyber security function should be independent from the information technology function. To
avoid any conflict of interest, the cyber security function a

In [10]:
!cd {KB} && python oscal.py flatten
!cd {KB} && python oscal.py families

catalog : Electronic (OSCAL) Version of NIST SP 800-53 Rev 5.2.0 Controls and SP 800-53A Rev 5.2.0 Assessment Procedures
version : 5.2.0   oscal 1.2.2
modified: 2026-05-11T16:01:09.00000-00:00

families            : 20
base controls       : 300
enhancements        : 714
active total        : 1014
withdrawn (excluded): 182
with ODPs           : 679 (66% of active)
total ODPs          : 1600

wrote 1196 rows -> processed/controls.jsonl

Withdrawn controls are kept in the file but flagged. They are
tombstones in Rev.5 and must never be offered as mapping targets.
The family router selects from this vocabulary.

      family  base   enh  ODPs  title
   1. AC        23   108   268  Access Control
   2. AT         5    10    34  Awareness and Training
   3. AU        15    41    76  Audit and Accountability
   4. CA         8    17    45  Assessment, Authorization, and Monitoring
   5. CM        14    42   108  Configuration Management
   6. CP        12    37    65  Contingency Planning
   

In [9]:
!ls -la {KB}/*.py

-rw------- 1 root root 28412 Jul 30 11:59 /content/drive/MyDrive/sanad-ai-readiness/kb/fetch.py
-rw------- 1 root root 14246 Jul 30 12:14 /content/drive/MyDrive/sanad-ai-readiness/kb/index.py
-rw------- 1 root root 11987 Jul 30 12:30 /content/drive/MyDrive/sanad-ai-readiness/kb/oscal.py


In [12]:
!cd {KB} && python sama.py parse
!cd {KB} && python sama.py tree 3.1.1

domains          : 4
subdomains       : 36
clauses total    : 493

by level
  level 1         160
  level 2         229
  level 3          90
  level 4          14

by type
  substantive     409
  stem             79
  referential       5

mappable units   : 488  (referential excluded)
leaf units       : 407  (the mapping source set)

deontic          : should=459, None=31, may=3
stating a value  : 7 clause(s)

subdomains with referential content: 3.3.12

wrote 493 clauses -> processed/sama_clauses.jsonl
3.1.1 Cyber Security Governance   (Cyber Security Leadership and Governance)

Principle : A cyber security governance structure should be defined and implemented, and should be endorsed by the board.
Objective : To direct and control the overall approach to cyber security within the Member Organization.

  1. A cyber security committee should be established and be mandated by the board.
     [3.1.1-1]
  2. The cyber security committee should be headed by an independent senior manager f

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
KB = '/content/drive/MyDrive/sanad-ai-readiness/kb'
os.environ['HF_HOME'] = '/content/drive/MyDrive/sanad-ai-readiness/.hf'

!pip install -q sentence-transformers pdfplumber pypdf

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 129.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 96.3 MB/s eta 0:00:00


In [2]:
!ls -la {KB}/*.py {KB}/processed/

-rw------- 1 root root 28412 Jul 30 11:59 /content/drive/MyDrive/sanad-ai-readiness/kb/fetch.py
-rw------- 1 root root 14246 Jul 30 12:14 /content/drive/MyDrive/sanad-ai-readiness/kb/index.py
-rw------- 1 root root 11987 Jul 30 12:30 /content/drive/MyDrive/sanad-ai-readiness/kb/oscal.py
-rw------- 1 root root 16570 Jul 31 08:10 /content/drive/MyDrive/sanad-ai-readiness/kb/router.py
-rw------- 1 root root 18251 Jul 30 12:39 /content/drive/MyDrive/sanad-ai-readiness/kb/sama.py

/content/drive/MyDrive/sanad-ai-readiness/kb/processed/:
total 4996
-rw------- 1 root root  949726 Jul 30 12:03 chunks.jsonl
-rw------- 1 root root 3609194 Jul 30 12:32 controls.jsonl
-rw------- 1 root root  551739 Jul 30 12:40 sama_clauses.jsonl
drwx------ 2 root root    4096 Jul 30 11:45 text


In [3]:
!cd {KB} && python router.py check

scoring 241 clauses across 21 anchored subdomains

modules.json: 100% 349/349 [00:00<00:00, 1.31MB/s]
config_sentence_transformers.json: 100% 124/124 [00:00<00:00, 222kB/s]
README.md: 100% 94.8k/94.8k [00:00<00:00, 53.2MB/s]
sentence_bert_config.json: 100% 52.0/52.0 [00:00<00:00, 298kB/s]
config.json: 100% 743/743 [00:00<00:00, 4.02MB/s]

model.safetensors: downloading bytes:  57% 75.5M/133M [00:02<00:00, 70.3MB/s, 5.54MB/s  ]
model.safetensors: downloading bytes: 100% 86.7M/86.7M [00:02<00:00, 35.8MB/s, 8.13MB/s  ]
model.safetensors: reconstructing file: 100% 133M/133M [00:02<00:00, 55.1MB/s, 12.7MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 3017.60it/s]
tokenizer_config.json: 100% 366/366 [00:00<00:00, 2.29MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 61.5MB/s]
tokenizer.json: 100% 711k/711k [00:00<00:00, 90.8MB/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 487kB/s]
config.json: 100% 190/190 [00:00<00:00, 802kB/s]
  built (20, 384) family profile embeddings
Batches: 10

In [4]:
!cd {KB} && python router.py route --k 5

routing 407 leaf clauses of 493 total
Loading weights: 100% 199/199 [00:00<00:00, 2914.75it/s]
Batches: 100% 7/7 [00:00<00:00, 11.06it/s]

family selection frequency (top-5)
  PM    390  ########################################
  SA    329  #################################
  SI    205  #####################
  RA    193  ###################
  SC    156  ################
  SR    135  #############
  PL    118  ############
  PS    116  ###########
  AC     90  #########
  PT     54  #####
  IA     46  ####
  PE     46  ####
  CM     45  ####
  AU     41  ####
  IR     26  ##
  CA     23  ##
  AT     17  #
  MP      5  #

  never selected: CP, MA
  (worth checking — a family no clause routes to is either
   genuinely absent from SAMA, or a routing blind spot)

candidate pairs after routing : 128,487
without routing               : 412,698
reduction                     : 69%

wrote 407 routes -> processed/routes.jsonl


In [5]:
!cd {KB} && python candidates.py check

using routes from routes.jsonl
note: 1 anchor clause(s) are not leaf clauses and were skipped: 3.1.7-1

indexing 1014 active controls (182 withdrawn excluded)
Loading weights: 100% 199/199 [00:00<00:00, 7429.76it/s]
  encoding 1014 controls
Batches: 100% 32/32 [00:04<00:00,  6.62it/s]

  N   recall   hits/total
  3   38.5%   5/13
  5   61.5%   8/13
 10   76.9%   10/13
 20   76.9%   10/13

per-clause at N=10
  3.1.6-1            want AT-2         rank 7   got AT-2(6), AT-2(1), RA-3(3), PM-16
  3.3.1-3.d          want PS-3         rank 8   got SA-8(27), PS-3(4), PS-9, SA-2
  3.3.2-3.a          want PE-3         rank 3   got PE-3(1), PE-3(2), PE-3, PE-2(1)
  3.3.5-4.b.6        want AC-2         rank 3   got AC-24(2), IA-8(4), AC-2, IA-12
  3.3.5-4.f.1.a      want AC-17,IA-2   rank 4   got IA-2(6), IA-2(1), IA-2(2), AC-17
  3.3.8-6.e          want SC-7         rank 4   got SA-8(3), SA-15(1), SC-17, SC-7
  3.3.9-4.c          want SC-12        rank 1   got SC-12, SC-13, SC-12(6), SC-28(3)
  

In [9]:
!cd {KB} && python candidates.py check
!cd {KB} && python candidates.py --base-quota 0 check

using routes from routes.jsonl
note: 1 anchor clause(s) are not leaf clauses and were skipped: 3.1.7-1

indexing 1014 active controls (182 withdrawn excluded)
  714 enhancements, 300 base controls
  reusing cached control embeddings (1014, 384)
Loading weights: 100% 199/199 [00:00<00:00, 5583.43it/s]

  N   recall   hits/total
  3   76.9%   10/13
  5   76.9%   10/13
 10   92.3%   12/13
 20   92.3%   12/13

per-clause at N=10
  3.1.6-1            want AT-2         rank 2   got PM-16, AT-2, AT-3, AT-2(6)
  3.3.1-3.d          want PS-3         rank 1   got PS-3, SA-9, SA-15, SA-8(27)
  3.3.2-3.a          want PE-3         rank 1   got PE-3, PE-6, PE-5, PE-8
  3.3.5-4.b.6        want AC-2         rank 1   got AC-2, IA-12, AC-21, IA-13
  3.3.5-4.f.1.a      want AC-17,IA-2   rank 1   got AC-17, AC-2, IA-12, IA-13
  3.3.8-6.e          want SC-7         rank 1   got SC-7, SI-4, SA-15, RA-5
  3.3.9-4.c          want SC-12        rank 1   got SC-12, SC-28, SC-13, SC-12(6)
  3.3.11-5           wa

In [10]:
!cd {KB} && python candidates.py generate --n 10

using routes from routes.jsonl
indexing 1014 active controls (182 withdrawn excluded)
  714 enhancements, 300 base controls
  reusing cached control embeddings (1014, 384)
Loading weights: 100% 199/199 [00:00<00:00, 3076.93it/s]
Batches: 100% 7/7 [00:01<00:00,  6.95it/s]

clauses          : 407
shortlist size   : 10
candidate pairs  : 4,070
controls reached : 356 of 1014 (35%)

most frequently shortlisted
  SA-9          157  External System Services
  RA-3(3)       117  Dynamic Threat Awareness
  SA-24         116  Design For Cyber Resiliency
  SI-4          112  System Monitoring
  SA-15          86  Development Process, Standards, and Tools
  RA-5           83  Vulnerability Monitoring and Scanning
  RA-3           80  Risk Assessment
  RA-3(4)        79  Predictive Cyber Analytics
  SA-3           70  System Development Life Cycle
  PM-16          68  Threat Awareness Program

658 control(s) never shortlisted. Some are genuinely
absent from SAMA — that set is the raw material for t

In [11]:
!cd {KB} && python judge.py dry-run 3.3.13-4.b.6.d

SYSTEM
You are a regulatory analyst mapping the SAMA Cyber Security Framework (Saudi Arabia, 2017) onto NIST SP 800-53 Rev 5.2.0.

You assign one of five relationship types, defined as in NIST IR 8477 with the SAMA clause as the focal element:

- equal: the two state the same requirement at the same scope.
- subset_of: the SAMA clause is narrower. The NIST control covers everything the SAMA clause requires, and more.
- superset_of: the SAMA clause is broader. It covers everything the NIST control requires, and more.
- intersects_with: they overlap in part; neither fully contains the other.
- not_related: no meaningful relationship.

Rules you must follow exactly:

1. Report only controls that stand in a real relationship. Omit the rest; omission means not_related.
2. Every judgment must quote evidence VERBATIM from the text given to you — an exact character-for-character substring, no paraphrase, no ellipsis, no correction of typography. Quote at least 15 characters from each side.
3. 

In [12]:
!cd {KB} && python oscal.py flatten
!cd {KB} && python oscal.py show SC-18

catalog : Electronic (OSCAL) Version of NIST SP 800-53 Rev 5.2.0 Controls and SP 800-53A Rev 5.2.0 Assessment Procedures
version : 5.2.0   oscal 1.2.2
modified: 2026-05-11T16:01:09.00000-00:00

families            : 20
base controls       : 300
enhancements        : 714
active total        : 1014
withdrawn (excluded): 182
with ODPs           : 679 (66% of active)
total ODPs          : 1600

wrote 1196 rows -> processed/controls.jsonl

Withdrawn controls are kept in the file but flagged. They are
tombstones in Rev.5 and must never be offered as mapping targets.
SC-18 — Mobile Code
family    : SC (System and Communications Protection)
type      : base control

statement :
a. Define acceptable and unacceptable mobile code and mobile code technologies; and
b. Authorize, monitor, and control the use of mobile code within the system.

related   : AU-2, AU-12, CM-2, CM-6, SI-3


In [13]:
!rm -f {KB}/index/controls.npy {KB}/index/controls_meta.json
!cd {KB} && python candidates.py check

using routes from routes.jsonl
note: 1 anchor clause(s) are not leaf clauses and were skipped: 3.1.7-1

indexing 1014 active controls (182 withdrawn excluded)
  714 enhancements, 300 base controls
Loading weights: 100% 199/199 [00:00<00:00, 2779.16it/s]
  encoding 1014 controls
Batches: 100% 32/32 [00:01<00:00, 20.08it/s]

  N   recall   hits/total
  3   69.2%   9/13
  5   76.9%   10/13
 10   76.9%   10/13
 20   84.6%   11/13

per-clause at N=10
  3.1.6-1            want AT-2         rank 1   got AT-2, AT-3, PM-16, SA-8
  3.3.1-3.d          want PS-3         rank 1   got PS-3, SA-15, SA-4, SA-9
  3.3.2-3.a          want PE-3         rank 1   got PE-3, PE-6, PE-8, PE-2
  3.3.5-4.b.6        want AC-2         rank 1   got AC-2, IA-13, IA-12, CM-5
  3.3.5-4.f.1.a      want AC-17,IA-2   rank 2   got AC-2, AC-17, IA-13, IA-12
  3.3.8-6.e          want SC-7         rank 4   got SA-15, SA-8, SC-16, SC-7
  3.3.9-4.c          want SC-12        rank 1   got SC-12, SC-13, IA-13(1), SC-28(3)
  3.3.

In [16]:
!rm -f {KB}/index/controls*.npy {KB}/index/controls*.json
!cd {KB} && python candidates.py --with-guidance check

using routes from routes.jsonl
note: 1 anchor clause(s) are not leaf clauses and were skipped: 3.1.7-1

  indexing statement + discussion
indexing 1014 active controls (182 withdrawn excluded)
  714 enhancements, 300 base controls
Loading weights: 100% 199/199 [00:00<00:00, 2772.84it/s]
  encoding 1014 controls
Batches: 100% 32/32 [00:04<00:00,  6.88it/s]

  N   recall   hits/total
  3   76.9%   10/13
  5   84.6%   11/13
 10   84.6%   11/13
 20   84.6%   11/13

per-clause at N=10
  3.1.6-1            want AT-2         rank 1   got AT-2, AT-3, PM-16, AT-2(6)
  3.3.1-3.d          want PS-3         rank 1   got PS-3, SA-15, SA-4, SA-3
  3.3.2-3.a          want PE-3         rank 1   got PE-3, PE-6, PE-2, PE-8
  3.3.5-4.b.6        want AC-2         rank 1   got AC-2, IA-13, IA-2, IA-12
  3.3.5-4.f.1.a      want AC-17,IA-2   rank 1   got IA-2, AC-2, AC-17, IA-13
  3.3.8-6.e          want SC-7         rank 4   got SI-5, SA-15, SA-9, SC-7
  3.3.9-4.c          want SC-12        rank 1   got SC-

In [17]:
!cd {KB} && python candidates.py --with-guidance show 3.3.13-4.b.6.d --n 20
!cd {KB} && python candidates.py --with-guidance show 3.3.13-11.c.2 --n 10

using routes from routes.jsonl
  indexing statement + discussion
indexing 1014 active controls (182 withdrawn excluded)
  714 enhancements, 300 base controls
  reusing cached control embeddings (1014, 384)
Loading weights: 100% 199/199 [00:00<00:00, 2916.87it/s]

3.3.13-4.b.6.d  (3.3.13 Electronic Banking Services)
revoking the access of customers after 3 successive incorrect passwords or invalid PINs;

states a value: 3 successive

routed families: SA, AC, SC, SI, PM

   1. SA-9         External System Services                       (both)  2 ODP
   2. SC-8         Transmission Confidentiality and Integrity     (both)  1 ODP
   3. AC-19        Access Control for Mobile Devices              (both)
   4. SC-18        Mobile Code                                    (both)
   5. SI-5         Security Alerts, Advisories, and Directives    (both)  5 ODP
   6. SA-8(28)     Acceptable Security                            (both)  1 ODP
   7. SA-4         Acquisition Process                      

In [18]:
!cd {KB} && python candidates.py --with-guidance --query-mode context check
!cd {KB} && python candidates.py --with-guidance --query-mode parent  check
!cd {KB} && python candidates.py --with-guidance --query-mode clause  check

using routes from routes.jsonl
note: 1 anchor clause(s) are not leaf clauses and were skipped: 3.1.7-1

  indexing statement + discussion
indexing 1014 active controls (182 withdrawn excluded)
  714 enhancements, 300 base controls
  reusing cached control embeddings (1014, 384)
Loading weights: 100% 199/199 [00:00<00:00, 2750.18it/s]

  N   recall   hits/total
  3   76.9%   10/13
  5   84.6%   11/13
 10   84.6%   11/13
 20   84.6%   11/13

per-clause at N=10
  3.1.6-1            want AT-2         rank 1   got AT-2, AT-3, PM-16, AT-2(6)
  3.3.1-3.d          want PS-3         rank 1   got PS-3, SA-15, SA-4, SA-3
  3.3.2-3.a          want PE-3         rank 1   got PE-3, PE-6, PE-2, PE-8
  3.3.5-4.b.6        want AC-2         rank 1   got AC-2, IA-13, IA-2, IA-12
  3.3.5-4.f.1.a      want AC-17,IA-2   rank 1   got IA-2, AC-2, AC-17, IA-13
  3.3.8-6.e          want SC-7         rank 4   got SI-5, SA-15, SA-9, SC-7
  3.3.9-4.c          want SC-12        rank 1   got SC-12, SI-7, SC-13, SC-16

In [19]:
!cd {KB} && python candidates.py --with-guidance generate --n 10

using routes from routes.jsonl
  indexing statement + discussion
indexing 1014 active controls (182 withdrawn excluded)
  714 enhancements, 300 base controls
  reusing cached control embeddings (1014, 384)
Loading weights: 100% 199/199 [00:00<00:00, 7256.58it/s]
Batches: 100% 7/7 [00:00<00:00, 11.98it/s]

clauses          : 407
shortlist size   : 10
candidate pairs  : 4,070
controls reached : 455 of 1014 (44%)

most frequently shortlisted
  SA-4          124  Acquisition Process
  SA-9          124  External System Services
  SA-24         120  Design For Cyber Resiliency
  RA-3(3)       104  Dynamic Threat Awareness
  SA-15          83  Development Process, Standards, and Tools
  SI-5           75  Security Alerts, Advisories, and Directives
  SA-11          70  Developer Testing and Evaluation
  SA-9(1)        65  Risk Assessments and Organizational Approvals
  SA-8(28)       61  Acceptable Security
  RA-3           60  Risk Assessment

559 control(s) never shortlisted. Some are genu

In [20]:
import json
seen = {c['control_id'] for l in open(f'{KB}/processed/candidates.jsonl',encoding='utf-8')
        for c in json.loads(l)['candidates']}
ctrl = [json.loads(l) for l in open(f'{KB}/processed/controls.jsonl',encoding='utf-8')]
live = [c for c in ctrl if not c['withdrawn']]
never = [c for c in live if c['control_id'] not in seen]

from collections import Counter
print('لم تظهر قط:', len(never))
for f,n in Counter(c['family'] for c in never).most_common():
    tot = sum(1 for c in live if c['family']==f)
    print(f'  {f:<4} {n:>3}/{tot:<4} ({100*n//tot}%)')

json.dump([c['control_id'] for c in never],
          open(f'{KB}/processed/never_shortlisted.json','w'), indent=1)

لم تظهر قط: 559
  AC    78/131  (59%)
  SC    61/139  (43%)
  SI    57/102  (55%)
  CP    49/49   (100%)
  AU    41/56   (73%)
  CM    35/56   (62%)
  IA    35/59   (59%)
  SA    35/108  (32%)
  PE    34/51   (66%)
  MA    28/28   (100%)
  MP    20/20   (100%)
  PT    16/21   (76%)
  IR    15/40   (37%)
  CA    14/25   (56%)
  SR    10/27   (37%)
  RA     9/22   (40%)
  AT     7/15   (46%)
  PM     7/37   (18%)
  PL     6/11   (54%)
  PS     2/17   (11%)


In [21]:
!cd {KB} && python judge.py dry-run 3.3.13-4.b.6.d | tail -3


approx input tokens: 2,326


In [22]:
!cd {KB} && python judge.py judge --mock --limit 20
!cd {KB} && python judge.py report
!rm -f {KB}/processed/findings.jsonl

judging 20 clause(s) with MOCK, tau=0.6

  10/20  accepted 10 disputed 10 rejected 10  eta 0.0 min
  20/20  accepted 20 disputed 20 rejected 20  eta 0.0 min

accepted 20 | disputed 20 | rejected 20 | clauses with no relation 0
parameter-gap findings: 2

Rejections are kept, not discarded. They are the evidence that
the gate does something, and their reasons are worth reporting.
JUDGMENT REPORT
clauses judged   : 20
judgments        : 60
  accepted          20  (33%)
  disputed          20  (33%)
  rejected          20  (33%)
clauses with no relation found: 0

relationship mix (verified judgments)
  subset_of             20
  intersects_with       20

distinct controls matched : 6
by family
  SI      15
  PL       3
  CA       1
  SC       1

parameter-gap findings: 1
  SI-5         external organizations

why the gate rejected (20)
     20  NIST evidence is not a verbatim span of the control

clauses with at least one accepted finding: 20/20 (100%)

A clause with no accepted finding is

In [23]:
!rm -f {KB}/processed/findings.jsonl

In [24]:
!pip install -q anthropic

!cd {KB} && python judge.py judge --subdomain 3.3.5 --limit 10
!cd {KB} && python judge.py show 3.3.5-4.f.1.a
!cd {KB} && python judge.py report

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.9 MB/s eta 0:00:00
Set ANTHROPIC_API_KEY (env var, or Colab secret of that name).
Run `python judge.py judge` first.
Run `python judge.py judge` first.


In [25]:
from google.colab import userdata
for n in ['ANTHROPIC_API_KEY','ANTHROPIC_API_KEY ','ANTHROPIC_KEY','ANTHROPIC_APIKEY']:
    try:
        v = userdata.get(n)
        print(f'✓ وُجد: {n!r}  الطول {len(v)}')
    except Exception as e:
        print(f'✗ {n!r}: {type(e).__name__}')

✓ وُجد: 'ANTHROPIC_API_KEY'  الطول 108
✗ 'ANTHROPIC_API_KEY ': ValueError
✗ 'ANTHROPIC_KEY': SecretNotFoundError
✗ 'ANTHROPIC_APIKEY': SecretNotFoundError


In [26]:
import os
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

!cd {KB} && python judge.py judge --subdomain 3.3.5 --limit 10
!cd {KB} && python judge.py show 3.3.5-4.f.1.a
!cd {KB} && python judge.py report

judging 10 clause(s) with anthropic:claude-sonnet-5, tau=0.6

  10/10  accepted 0 disputed 0 rejected 0  eta 0.0 min

accepted 0 | disputed 0 | rejected 0 | clauses with no relation 10
no findings recorded for 3.3.5-4.f.1.a
JUDGMENT REPORT
clauses judged   : 10
judgments        : 0
  accepted           0  (0%)
  disputed           0  (0%)
  rejected           0  (0%)
clauses with no relation found: 10

relationship mix (verified judgments)

distinct controls matched : 0
by family

clauses with at least one accepted finding: 0/10 (0%)

A clause with no accepted finding is a coverage gap candidate — but
check whether it is referential before counting it as one.


In [27]:
import json
for l in open(f'{KB}/processed/findings.jsonl', encoding='utf-8'):
    r = json.loads(l)
    print(r['clause_id'], '|', repr(r.get('parse_error'))[:200])

3.3.5-1 | "BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': '`temperature` is deprecated for this model.'}, 'request_id': 'req_011CdZubCnDwhMz3gVQtYTq2'
3.3.5-2 | "BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': '`temperature` is deprecated for this model.'}, 'request_id': 'req_011CdZubEgaUGzukwev4LQ2Z'
3.3.5-3 | "BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': '`temperature` is deprecated for this model.'}, 'request_id': 'req_011CdZubH4CjCQVcLzVBH24C'
3.3.5-4.a | "BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': '`temperature` is deprecated for this model.'}, 'request_id': 'req_011CdZubKQavEgK9jrvKWpG6'
3.3.5-4.b.1 | "BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': '`temperature` is deprecated for

In [28]:
!rm -f {KB}/processed/findings.jsonl
!cd {KB} && python judge.py judge --subdomain 3.3.5 --limit 10
!cd {KB} && python judge.py errors
!cd {KB} && python judge.py show 3.3.5-4.f.1.a
!cd {KB} && python judge.py report

judging 10 clause(s) with anthropic:claude-sonnet-5, tau=0.6

  ! 3.3.5-4.b.3: malformed JSON: Expecting value: line 1 column 2 (char 1)
  10/10  accepted 9 disputed 7 rejected 0  eta 0.0 min

accepted 9 | disputed 7 | rejected 0 | no relation 0 | CALLS FAILED 1

*** Failed calls are not findings. Nothing was judged for those
*** clauses. Fix the cause, delete findings.jsonl, and re-run.
*** Inspect the reasons with:  python judge.py errors
parameter-gap findings: 4
1 of 10 clause(s) failed

     1  malformed JSON

first few:
  3.3.5-4.b.3        malformed JSON: Expecting value: line 1 column 2 (char 1)
no findings recorded for 3.3.5-4.f.1.a
unparseable response. Those are not findings. See `judge.py errors`.

JUDGMENT REPORT
clauses judged   : 10
judgments        : 16
  accepted           9  (56%)
  disputed           7  (43%)
  rejected           0  (0%)
clauses with no relation found: 1

relationship mix (verified judgments)
  intersects_with       10
  subset_of              4
  eq

In [29]:
!cd {KB} && python judge.py show 3.3.5-4.f.1.a
!cd {KB} && python judge.py show 3.3.5-1

no findings recorded for 3.3.5-4.f.1.a
3.3.5-1   (3.3.5)
The identity and access management policy, including the responsibilities and accountabilities, should be defined, approved and implemented.

[+] AC-1         equal            conf 0.85
      SAMA: “The identity and access management policy, including the responsibilities and accountabilities, should be defi”
      NIST: “Develop, document, and disseminate to [organization-defined personnel or roles]:
1. [[select one-or-more: orga”
      Both require a defined, approved access/identity management policy with assigned responsibilities and accounta



In [30]:
import json
from collections import Counter
rows=[json.loads(l) for l in open(f'{KB}/processed/findings.jsonl',encoding='utf-8')]
fs=[f for r in rows for f in r['findings']]

print(Counter(f['relationship'] for f in fs), '\n')
for r in rows:
    for f in r['findings']:
        print(f"{r['clause_id']:<14} {f['control_id']:<8} {f['relationship']:<16} "
              f"{f['confidence']}  {f['status']}")
        print(f"   {f['rationale'][:105]}")

Counter({'intersects_with': 10, 'subset_of': 4, 'equal': 1, 'superset_of': 1}) 

3.3.5-1        AC-1     equal            0.85  accepted
   Both require a defined, approved access/identity management policy with assigned responsibilities and acc
3.3.5-2        AC-2     intersects_with  0.65  accepted
   AC-2 monitoring of account usage overlaps with SAMA's requirement to monitor compliance with identity and
3.3.5-2        AC-1     intersects_with  0.4  disputed
   AC-1 requires periodic review/update of the access control policy itself, which is related but distinct f
3.3.5-3        PM-6     intersects_with  0.65  accepted
   Both concern measuring and monitoring performance/effectiveness of security controls, though PM-6 is broa
3.3.5-3        AC-2     intersects_with  0.4  disputed
   AC-2's periodic account review is one operational measure of IAM effectiveness, but does not cover overal
3.3.5-4.a      AC-1     intersects_with  0.6  accepted
   AC-1 requires an access control policy

In [31]:
!rm -f {KB}/processed/findings.jsonl
!cd {KB} && python judge.py judge --max-tokens 6000

judging 407 clause(s) with anthropic:claude-sonnet-5, tau=0.6

  10/407  accepted 1 disputed 6 rejected 0  eta 37.7 min
  20/407  accepted 3 disputed 8 rejected 0  eta 33.3 min
  30/407  accepted 3 disputed 21 rejected 1  eta 36.9 min
  40/407  accepted 4 disputed 31 rejected 1  eta 37.1 min
  50/407  accepted 5 disputed 34 rejected 1  eta 32.9 min
  60/407  accepted 7 disputed 39 rejected 1  eta 31.2 min
  70/407  accepted 8 disputed 47 rejected 1  eta 31.2 min
  80/407  accepted 9 disputed 51 rejected 1  eta 29.2 min
  90/407  accepted 12 disputed 57 rejected 1  eta 27.9 min
  100/407  accepted 18 disputed 67 rejected 1  eta 27.2 min
  110/407  accepted 25 disputed 69 rejected 1  eta 25.7 min
  120/407  accepted 27 disputed 74 rejected 2  eta 25.3 min
  130/407  accepted 31 disputed 77 rejected 2  eta 24.0 min
  140/407  accepted 34 disputed 84 rejected 2  eta 23.0 min
  150/407  accepted 35 disputed 88 rejected 2  eta 22.0 min
  160/407  accepted 37 disputed 96 rejected 2  eta 21.0 

In [32]:
import json
from collections import Counter
rows=[json.loads(l) for l in open(f'{KB}/processed/findings.jsonl',encoding='utf-8')]
cl={c['clause_id']:c for c in (json.loads(l) for l in open(f'{KB}/processed/sama_clauses.jsonl',encoding='utf-8'))}

pg=[(r['clause_id'],f) for r in rows for f in r['findings']
    if f['parameter_gap'] and f['status']=='accepted']
hard=[x for x in pg if cl[x[0]]['has_numeric_param']]
print(f'ثغرات معامل مقبولة: {len(pg)}  منها بقيمة صريحة في ساما: {len(hard)}')
for cid,f in hard:
    print(f"  {cid:<18} {f['control_id']:<10} {f['parameter_note'][:70]}")

ثغرات معامل مقبولة: 49  منها بقيمة صريحة في ساما: 4
  3.2.4-2            CA-8       frequency (SAMA specifies annual)
  3.3.8-6.h.3        SI-4       SAMA specifies continuous 24x7 monitoring; NIST leaves frequency/monit
  3.3.8-6.h.4        SC-5       frequency of DDoS scrubbing testing (twice a year) not specified as pa
  3.3.13-4.b.6.d     AC-7       SAMA specifies 3 successive attempts; NIST leaves 'number' as organiza


In [33]:
!cd {KB} && python judge.py judge --subdomain 3.4.3 --max-tokens 8000
!cd {KB} && python judge.py errors

resuming: 404 clause(s) already judged
nothing to do.
1 of 407 clause(s) failed

     1  no JSON array in the response

first few:
  3.4.3-4.g.2        no JSON array in the response


In [35]:
!cd {KB} && python judge.py judge --subdomain 3.4.3 --max-tokens 8000
!cd {KB} && python judge.py errors
!cd {KB} && python judge.py report

dropped 1 failed record(s) for retry: 3.4.3-4.g.2
resuming: 403 clause(s) already judged
judging 1 clause(s) with anthropic:claude-sonnet-5, tau=0.6

  1/1  accepted 1 disputed 1 rejected 0  eta 0.0 min

accepted 1 | disputed 1 | rejected 0 | no relation 0 | CALLS FAILED 0
no call or parse errors in 407 record(s).
111 clause(s) returned an empty array — that is a real
answer from the model, not a failure.
JUDGMENT REPORT
clauses judged   : 407
judgments        : 541
  accepted         186  (34%)
  disputed         340  (62%)
  rejected          15  (2%)
clauses with no relation found: 111

relationship mix (verified judgments)
  intersects_with      321
  subset_of            165
  equal                 20
  superset_of           20

distinct controls matched : 82
by family
  SA      26
  SI      22
  RA      20
  AC      17
  IR      17
  PM      14
  SC      14
  AT      13
  CM      12
  IA       9
  PS       6
  PE       6
  CA       4
  PL       4
  AU       1
  SR       1

parame

In [36]:
import json
rows=[json.loads(l) for l in open(f'{KB}/processed/findings.jsonl',encoding='utf-8')]
au=[(r['clause_id'],f) for r in rows for f in r['findings'] if f['control_id'].startswith('AU')]
print('كل أحكام AU:', len(au))
for cid,f in au[:10]: print(f"  {cid:<16} {f['control_id']:<8} {f['status']:<10} {f['confidence']}")

كل أحكام AU: 8
  3.2.4-1          AU-6     disputed   0.3
  3.2.4-3          AU-6     disputed   0.4
  3.2.4-4          AU-6     disputed   0.4
  3.3.14-1         AU-2     disputed   0.45
  3.3.14-3.a       AU-2     accepted   0.72
  3.3.14-4.f       AU-2     disputed   0.4
  3.3.14-4.g       AU-2     disputed   0.4
  3.3.14-4.j       AU-7(1)  disputed   0.5


In [37]:
import json
from collections import defaultdict
rows=[json.loads(l) for l in open(f'{KB}/processed/findings.jsonl',encoding='utf-8')]
fam=defaultdict(lambda: {'acc':set(),'dis':set(),'conf':[]})
for r in rows:
    for f in r['findings']:
        if f['status']=='rejected': continue
        k=f['control_id'].split('-')[0]
        fam[k]['acc' if f['status']=='accepted' else 'dis'].add(f['control_id'])
        fam[k]['conf'].append(f['confidence'] or 0)

print(f"{'fam':<5}{'مقبول':>7}{'+متنازع':>9}{'مخفي':>7}{'متوسط الثقة':>13}")
for k in sorted(fam, key=lambda x:-len(fam[x]['acc']|fam[x]['dis'])):
    v=fam[k]; both=v['acc']|v['dis']
    print(f"{k:<5}{len(v['acc']):>7}{len(both):>9}{len(both)-len(v['acc']):>7}"
          f"{sum(v['conf'])/len(v['conf']):>13.2f}")

fam    مقبول  +متنازع   مخفي  متوسط الثقة
SC        11       21     10         0.53
SA         9       19     10         0.55
PM         7       18     11         0.53
SI        10       16      6         0.59
AC         6       14      8         0.59
CM         5       13      8         0.58
IA         5       11      6         0.59
IR         7       11      4         0.56
PS         5       10      5         0.58
PE         5       10      5         0.54
RA         3        8      5         0.58
CA         3        5      2         0.57
PL         1        4      3         0.53
AT         3        4      1         0.66
SR         1        4      3         0.47
AU         1        3      2         0.45
PT         0        2      2         0.47


In [38]:
!pip install -q gradio
!cd {KB} && python app.py selftest
!cd {KB} && python app.py

data ready: True 
  clauses 489  controls 1196 (1014 active)  findings rows 407
  accepted 186  verified 526
  clause gaps 249; first: 3.1.1-1
  control gaps: worst family CP 100%
  parameter gaps: hard 4 soft 45
  drift: {'SR': {'total': 27, 'matched': 1, 'title': 'Supply Chain Risk Management'}, 'PT': {'total': 21, 'matched': 0, 'title': 'Personally Identifiable Information Processing and Transparency'}, 'rev520': {'known': ['SA-15(13)', 'SA-24', 'SI-2(7)'], 'matched': ['SI-2(7)']}}
  clause choices 404
  detail rows for 3.1.1-1: 0
  kb rows 18
  itu factors 6 dimensions 13

## SANAD — evidence-anchored regulatory alignment

**SAMA Cyber Security Framework (2017)** against **NIST SP 800-53 Rev 5.2.0**,
with every claim anchored to a verbatim span of both sources.

| | |
|---|---|
| clauses analysed | **407** |
| judgments | **541** — 186 accepted, 340 for review, 15 rejected at the gate |
| clauses with an accepted match | **158** (38%) |
| coverage gap candidates | *
* Running on lo

In [39]:
import json
c=[json.loads(l) for l in open(f'{KB}/processed/candidates.jsonl',encoding='utf-8')
   if json.loads(l)['clause_id']=='3.1.1-1'][0]
print('العائلات:', c['families'])
print('المرشحون:', [x['control_id'] for x in c['candidates']])

f=[json.loads(l) for l in open(f'{KB}/processed/findings.jsonl',encoding='utf-8')
   if json.loads(l)['clause_id']=='3.1.1-1'][0]
print('الأحكام:', f['findings'])
print('خطأ التحليل:', repr(f.get('parse_error')))

العائلات: ['PM', 'SA', 'PL', 'RA', 'SI']
المرشحون: ['SI-5', 'SA-4', 'SA-24', 'PM-23', 'PM-24', 'PM-3', 'SI-5(1)', 'SI-1', 'SA-8(28)', 'SI-4(17)']
الأحكام: []
خطأ التحليل: ''


In [40]:
!cd {KB} && python oscal.py show PM-2
!cd {KB} && python oscal.py show PM-1

PM-2 — Information Security Program Leadership Role
family    : PM (Program Management)
type      : base control

statement :
Appoint a senior agency information security officer with the mission and resources to coordinate, develop, implement, and maintain an organization-wide information security program.
PM-1 — Information Security Program Plan
family    : PM (Program Management)
type      : base control

statement :
a. Develop and disseminate an organization-wide information security program plan that:
1. Provides an overview of the requirements for the security program and a description of the security program management controls and common controls in place or planned for meeting those requirements;
2. Includes the identification and assignment of roles, responsibilities, management commitment, coordination among organizational entities, and compliance;
3. Reflects the coordination among organizational entities responsible for information security; and
4. Is approved by a senior 

In [41]:
import json
for cid in ['3.1.1-8','3.1.1-1','3.1.1-5']:
    r=[json.loads(l) for l in open(f'{KB}/processed/findings.jsonl',encoding='utf-8')
       if json.loads(l)['clause_id']==cid][0]
    print(cid, '→', [(f['control_id'],f['relationship'],f['confidence'],f['status'])
                     for f in r['findings']] or 'لا شيء')

3.1.1-8 → [('PM-2', 'equal', 0.85, 'accepted')]
3.1.1-1 → لا شيء
3.1.1-5 → لا شيء


In [42]:
!cd {KB} && python app.py selftest

data ready: True 
  clauses 489  controls 1196 (1014 active)  findings rows 407
  accepted 186  verified 526
  clause gaps 249; first: 3.1.4-1.b
  weakest subdomain: 3.2.1.3 100% (16/16)
  mapping opens on: 3.3.8-6.g
  control gaps: worst family CP 100%
  parameter gaps: hard 4 soft 45
  drift: {'SR': {'total': 27, 'matched': 1, 'title': 'Supply Chain Risk Management'}, 'PT': {'total': 21, 'matched': 0, 'title': 'Personally Identifiable Information Processing and Transparency'}, 'rev520': {'known': ['SA-24', 'SI-2(7)', 'SA-15(13)'], 'matched': ['SI-2(7)']}}
  clause choices 404
  detail rows for 3.1.1-1: 0
  kb rows 18
  itu factors 6 dimensions 13

## SANAD — evidence-anchored regulatory alignment

**SAMA Cyber Security Framework (2017)** against **NIST SP 800-53 Rev 5.2.0**,
with every claim anchored to a verbatim span of both sources.

| | |
|---|---|
| clauses analysed | **407** |
| judgments | **541** — 186 accepted, 340 for review, 15 rejected at the gate |
| clauses with an acce

In [43]:
import json
c=[json.loads(l) for l in open(f'{KB}/processed/candidates.jsonl',encoding='utf-8')
   if json.loads(l)['clause_id'].startswith('3.2.1.3-')]
print('عدد البنود:', len(c))
for x in c[:4]:
    print(x['clause_id'], '| العائلات:', x['families'])
    print('   ', [k['control_id'] for k in x['candidates']][:6])

عدد البنود: 16
3.2.1.3-1 | العائلات: ['RA', 'PM', 'SA', 'PS', 'SI']
    ['RA-3', 'SI-5', 'SA-4', 'RA-2', 'SA-3', 'SA-24']
3.2.1.3-2 | العائلات: ['RA', 'PM', 'SA', 'SR', 'SC']
    ['SR-2', 'RA-3', 'SR-3', 'SR-5', 'SA-3', 'SA-9']
3.2.1.3-3.a | العائلات: ['RA', 'PM', 'SA', 'SC', 'SR']
    ['SR-2', 'RA-3', 'SA-4', 'SA-24', 'RA-3(3)', 'RA-7']
3.2.1.3-3.b.1 | العائلات: ['RA', 'PM', 'SA', 'SR', 'PS']
    ['SR-2', 'RA-3', 'SR-6', 'PS-6', 'SA-4', 'RA-3(3)']


In [44]:
import json
from collections import Counter
rows=[json.loads(l) for l in open(f'{KB}/processed/findings.jsonl',encoding='utf-8')
      if json.loads(l)['clause_id'].startswith('3.2.1.3-')]
print(Counter(f['status'] for r in rows for f in r['findings']))
print('بنود بلا أي حكم:', sum(1 for r in rows if not r['findings']))
for r in rows[:6]:
    print(r['clause_id'], [(f['control_id'],f['confidence'],f['status']) for f in r['findings']])

Counter({'disputed': 10})
بنود بلا أي حكم: 8
3.2.1.3-1 []
3.2.1.3-2 [('RA-3', 0.55, 'disputed')]
3.2.1.3-3.a [('RA-7', 0.55, 'disputed')]
3.2.1.3-3.b.1 [('RA-7', 0.55, 'disputed')]
3.2.1.3-3.b.2 []
3.2.1.3-4 [('RA-7', 0.55, 'disputed')]


In [45]:
import json
from collections import Counter
rows=[json.loads(l) for l in open(f'{KB}/processed/findings.jsonl',encoding='utf-8')]
confs=[f['confidence'] for r in rows for f in r['findings'] if f['status']!='rejected']

print('توزيع القيم:')
for v,n in sorted(Counter(confs).items()):
    print(f'  {v}: {n:>4}  {"#"*(n//10)}')

print('\nمنحنى المخاطرة–التغطية:')
for t in [0.4,0.45,0.5,0.55,0.6,0.65,0.7,0.8]:
    n=sum(1 for c in confs if c>=t)
    print(f'  tau={t}: {n:>3} مقبول  ({100*n//len(confs)}% تغطية)')

توزيع القيم:
  0.3:   10  #
  0.35:   25  ##
  0.4:   85  ########
  0.45:   29  ##
  0.5:   59  #####
  0.55:  132  #############
  0.6:   37  ###
  0.62:    3  
  0.65:   33  ###
  0.7:    6  
  0.72:    2  
  0.75:   58  #####
  0.78:    1  
  0.8:    3  
  0.82:    1  
  0.85:   35  ###
  0.88:    1  
  0.9:    5  
  0.95:    1  

منحنى المخاطرة–التغطية:
  tau=0.4: 491 مقبول  (93% تغطية)
  tau=0.45: 406 مقبول  (77% تغطية)
  tau=0.5: 377 مقبول  (71% تغطية)
  tau=0.55: 318 مقبول  (60% تغطية)
  tau=0.6: 186 مقبول  (35% تغطية)
  tau=0.65: 146 مقبول  (27% تغطية)
  tau=0.7: 113 مقبول  (21% تغطية)
  tau=0.8:  46 مقبول  (8% تغطية)


In [46]:
import json, random
random.seed(3)
rows=[json.loads(l) for l in open(f'{KB}/processed/findings.jsonl',encoding='utf-8')]
mid=[(r['clause_id'],f) for r in rows for f in r['findings'] if f['confidence']==0.55]
for cid,f in random.sample(mid,15):
    print(f"{cid:<18} {f['control_id']:<10} {f['relationship']:<16}")
    print(f"   {f['rationale'][:120]}\n")

3.3.5-4.d          IA-13      intersects_with 
   IA-13 mandates centralized identity providers/authorization servers as infrastructure, which relates to but does not equ

3.2.1.3-3.b.1      RA-7       intersects_with 
   Both address responding to risk in line with organizational risk tolerance/appetite, but SAMA specifically requires comm

3.3.13-11.c.2      AC-7       intersects_with 
   Both address limiting consecutive invalid authentication attempts before an automated response, though the SAMA clause t

3.4.1-5.f          SA-4(12)   intersects_with 
   Both address data handling upon contract termination, though SA-4(12) focuses narrowly on data return while SAMA covers 

3.1.4-4.g.2        SA-4       intersects_with 
   SA-4 covers determining security requirements but in acquisition contract context, while SAMA clause is about projects g

3.1.2-1            PM-9       intersects_with 
   Both address development, implementation, and ongoing maintenance of a strategic risk/secu

In [47]:
!cd {KB} && python app.py selftest
!cd {KB} && python app.py

data ready: True 
  clauses 489  controls 1196 (1014 active)  findings rows 407
  accepted 186  verified 526
  clause gaps 249; first: 3.1.4-1.b
  weakest subdomain: 3.1.4 — 22 no match, 12 for review, of 39  [no match found]
  mapping opens on: 3.3.8-6.g
  control gaps: worst family CP 100%
  parameter gaps: hard 4 soft 45
  drift: {'SR': {'total': 27, 'matched': 1, 'title': 'Supply Chain Risk Management'}, 'PT': {'total': 21, 'matched': 0, 'title': 'Personally Identifiable Information Processing and Transparency'}, 'rev520': {'known': ['SI-2(7)', 'SA-15(13)', 'SA-24'], 'matched': ['SI-2(7)']}}
  clause choices 404
  detail rows for 3.1.1-1: 0
  kb rows 18
  itu factors 6 dimensions 13

## SANAD — evidence-anchored regulatory alignment

**SAMA Cyber Security Framework (2017)** against **NIST SP 800-53 Rev 5.2.0**,
with every claim anchored to a verbatim span of both sources.

| | |
|---|---|
| clauses analysed | **407** |
| judgments | **541** — 186 accepted, 340 for review, 15 reject

In [49]:
!cd {KB} && python app.py

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://c6ebe89ec2fa2ea32f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Keyboard interruption in main thread... closing server.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 3367, in block_thread
    time.sleep(0.1)
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/drive/MyDrive/sanad-ai-readiness/kb/app.py", line 672, in <module>
    sys.exit(selftest() if "selftest" in sys.argv else launch("--no-share" not in sys.argv))
                                                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/sanad-ai-rea

In [51]:
import shutil, pathlib, os
KB = '/content/drive/MyDrive/sanad-ai-readiness/kb'
S  = '/content/space'
shutil.rmtree(S, ignore_errors=True)
os.makedirs(f'{S}/processed'); os.makedirs(f'{S}/index')

for f in ['app.py','index.py','manifest.csv']:
    shutil.copy(f'{KB}/{f}', S)
for f in ['findings.jsonl','sama_clauses.jsonl','controls.jsonl',
          'candidates.jsonl','chunks.jsonl']:
    shutil.copy(f'{KB}/processed/{f}', f'{S}/processed/')
for f in ['dense.npy','dense_meta.json']:
    shutil.copy(f'{KB}/index/{f}', f'{S}/index/')

open(f'{S}/requirements.txt','w').write("gradio\nnumpy\nsentence-transformers\n")

open(f'{S}/README.md','w').write("""---
title: SANAD
emoji: 🔗
colorFrom: blue
colorTo: gray
sdk: gradio
app_file: app.py
pinned: false
---

# SANAD — Evidence-Anchored Regulatory Alignment

SAMA Cyber Security Framework (2017) against NIST SP 800-53 Rev 5.2.0.
Every finding is anchored to a verbatim span of both sources.

Source documents are not redistributed; the corpus is reproducible from
`manifest.csv`.
""")

!du -sh {S} && ls -R {S} | head -20

5.2M	/content/space
/content/space:
app.py
index
index.py
manifest.csv
processed
README.md
requirements.txt

/content/space/index:
dense_meta.json
dense.npy

/content/space/processed:
candidates.jsonl
chunks.jsonl
controls.jsonl
findings.jsonl
sama_clauses.jsonl


In [53]:
!pip install -q huggingface_hub
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

from huggingface_hub import HfApi
api = HfApi()
USER = api.whoami()['name']
REPO = f'{USER}/sanad-ai-readiness'

api.create_repo(REPO, repo_type='space', space_sdk='gradio', exist_ok=True)
api.upload_folder(folder_path=S, repo_id=REPO, repo_type='space')

print(f'https://huggingface.co/spaces/{REPO}')

HfHubHTTPError: Client error '402 Payment Required' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-6a6ca737-3c3158aa686644e713b0ca05;853169fc-83e6-4109-96e1-6dd08064ab79)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

Static Spaces are free for everyone, but hosting Gradio and Docker Spaces on free cpu-basic requires a PRO subscription. Subscribe at https://huggingface.co/pro

In [54]:
!cd {KB} && pip install -q pdfplumber

In [55]:
import pdfplumber
with pdfplumber.open(f'{KB}/raw/SAU-SDAIA-AIETHICS-2025.pdf') as pdf:
    for i, p in enumerate(pdf.pages[:12], 1):
        t = p.extract_text() or ''
        if any(w in t.lower() for w in ['principle', 'contents', 'fairness']):
            print(f'--- ص {i} ---\n{t[:900]}\n')

--- ص 1 ---
AI
Ethics Principles
2025

--- ص 3 ---
Table of Content
Introduction 04
Definitions 06
Scope 09
AI Ethics Principles and Controls 12
Roles and Responsibilities 28
Appendices 33

--- ص 4 ---
AI Ethics Principles
Introduction
Due to the fast growth of practices and technologies around Artificial Intelligence (AI), the use of AI has
expanded to several industries such as health, education, entertainment, etc. AI helps make entities dec-
sion-making processes more efficient, accurate and faster by predicting future patterns. AI can be used to
analyze data, including big data, by developing and operating systems with advanced models and algrithms
that help improve the quality of processes. Due to the increasing interest in these technologies, many enti-
ties in both the public and private sectors, in addition to non-profit entities, have developed digital solutions
based on AI to address existing challenges using creative and innovative methods, thus making the role of AI
more e

In [56]:
import pdfplumber, re
with pdfplumber.open(f'{KB}/raw/SAU-SDAIA-AIETHICS-2025.pdf') as pdf:
    for i, p in enumerate(pdf.pages[11:28], 12):
        for line in (p.extract_text() or '').split('\n'):
            if re.match(r'^\s*Principle\s+\d+', line):
                print(f'ص{i:>3}  {line.strip()}')

ص 12  Principle 1 – Fairness
ص 15  Principle 2 – Privacy & Security
ص 18  Principle 3 – Humanity
ص 20  Principle 4 – Social & Environmental Benefits
ص 22  Principle 5 – Reliability & Safety
ص 24  Principle 6 – Transparency & Explainability
ص 26  Principle 7 – Accountability & Responsibility


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
from google.colab import userdata
KB = '/content/drive/MyDrive/sanad-ai-readiness/kb'
os.environ['HF_HOME'] = '/content/drive/MyDrive/sanad-ai-readiness/.hf'
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
os.environ['OPENAI_API_KEY']    = userdata.get('OPENAI_API_KEY')

!pip install -q anthropic openai
!ls {KB}/*.py && ls {KB}/processed/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.5 MB/s eta 0:00:00
/content/drive/MyDrive/sanad-ai-readiness/kb/app.py
/content/drive/MyDrive/sanad-ai-readiness/kb/candidates.py
/content/drive/MyDrive/sanad-ai-readiness/kb/debate.py
/content/drive/MyDrive/sanad-ai-readiness/kb/fetch.py
/content/drive/MyDrive/sanad-ai-readiness/kb/index.py
/content/drive/MyDrive/sanad-ai-readiness/kb/judge.py
/content/drive/MyDrive/sanad-ai-readiness/kb/oscal.py
/content/drive/MyDrive/sanad-ai-readiness/kb/router.py
/content/drive/MyDrive/sanad-ai-readiness/kb/sama.py
candidates.jsonl  controls.jsonl  never_shortlisted.json  sama_clauses.jsonl
chunks.jsonl	  findings.jsonl  routes.jsonl		  text


In [3]:
!cd {KB} && python debate.py run --mock --limit 10
!cd {KB} && python debate.py report

debating 10 finding(s) from band 'provisional'
  opponent: MOCK
  arbiter : MOCK

  10/10  objections 10  changed 10  eta 0.0 min

objections raised : 10 of 10
verdicts          : revise=10
verdict changed the mapping: 10

relationship shifts
  intersects_with->subset_of               10

wrote -> processed/findings_v2.jsonl

findings.jsonl is untouched. Compare with `python debate.py compare`.
DEBATE REPORT
findings debated   : 10
objections raised  : 10 (100%)
mappings changed   : 10 (100%)

verdicts
  revise                 10

grounds of objection
  wrong_type             10

relationship before -> after (changed only)
  3.1.1-2            PM-29      intersects_with  -> subset_of
  3.1.1-4.b          SA-3       intersects_with  -> subset_of
  3.1.1-4.b          SA-9       intersects_with  -> subset_of
  3.1.1-10           SA-2       intersects_with  -> subset_of
  3.1.2-1            PM-9       intersects_with  -> subset_of
  3.1.3-1            SC-1       intersects_with  -> subset_

In [4]:
!rm -f {KB}/processed/findings_v2.jsonl

In [5]:
!cd {KB} && python debate.py run --band provisional --limit 20

debating 20 finding(s) from band 'provisional'
  opponent: openai:gpt-4o
  arbiter : anthropic:claude-sonnet-5

  10/20  objections 10  changed 8  eta 1.8 min
  20/20  objections 19  changed 12  eta 0.0 min

objections raised : 19 of 20
verdicts          : uphold=7, revise=12, reject=1
verdict changed the mapping: 12

relationship shifts
  intersects_with->subset_of               5
  intersects_with->intersects_with         3
  superset_of->subset_of                   2
  intersects_with->superset_of             1
  subset_of->intersects_with               1

wrote -> processed/findings_v2.jsonl

findings.jsonl is untouched. Compare with `python debate.py compare`.


In [6]:
import json
rows=[json.loads(l) for l in open(f'{KB}/processed/findings_v2.jsonl',encoding='utf-8')]
for r in rows:
    if r['verdict'] in ('reject','uphold') or r['original']['relationship']=='subset_of':
        o=r['objection']
        print(f"{r['clause_id']:<16} {r['control_id']:<9} "
              f"{r['original']['relationship']} → {r['final']['relationship']} [{r['verdict']}]")
        print(f"   اعتراض ({o.get('grounds','')}): {o.get('argument','')[:170]}")
        print(f"   حكم: {r['ruling'].get('reasoning','')[:170]}\n")

3.1.1-2          PM-29     intersects_with → intersects_with [uphold]
   اعتراض (wrong_type): SAMA requires independence from control functions, while PM-29 does not specify this. SAMA's clause is narrower, thus subset_of is more appropriate.
   حكم: SAMA adds a specific independence/control-function requirement not present in PM-29, while PM-29's org-wide risk executive scope covers more than just committee leadershi

3.1.1-4.b        SA-3      intersects_with → intersects_with [uphold]
   اعتراض (wrong_type): SA-3 is limited to SDLC roles while SAMA 3.1.1-4.b covers broader governance roles, making SAMA broader. They overlap without full containment.
   حكم: Both sides agree the relationship is intersects_with: SAMA's clause covers general governance roles beyond SDLC, while SA-3 covers SDLC-specific processes (e.g., risk man

3.1.3-2          PL-1      intersects_with → intersects_with [uphold]
   اعتراض (wrong_control): PL-1 does relate to periodic review of planning policy, applic

In [7]:
!cd {KB} && python debate.py run --band provisional
!cd {KB} && python debate.py compare
!cd {KB} && python debate.py report

resuming: 20 already debated
debating 244 finding(s) from band 'provisional'
  opponent: openai:gpt-4o
  arbiter : anthropic:claude-sonnet-5

  10/244  objections 9  changed 6  eta 39.4 min
  20/244  objections 18  changed 13  eta 32.7 min
  30/244  objections 26  changed 18  eta 33.0 min
  40/244  objections 34  changed 22  eta 31.8 min
  50/244  objections 43  changed 28  eta 29.3 min
  60/244  objections 51  changed 36  eta 26.7 min
  70/244  objections 60  changed 42  eta 25.6 min
  80/244  objections 70  changed 51  eta 23.4 min
  90/244  objections 79  changed 60  eta 21.8 min
  100/244  objections 88  changed 65  eta 20.1 min
  110/244  objections 98  changed 71  eta 19.4 min
  120/244  objections 107  changed 76  eta 18.0 min
  130/244  objections 116  changed 84  eta 16.3 min
  140/244  objections 126  changed 90  eta 14.6 min
  150/244  objections 135  changed 95  eta 13.3 min
  160/244  objections 144  changed 100  eta 11.8 min
  170/244  objections 153  changed 107  eta 10.

In [8]:
import json, random
random.seed(11)
rows=[json.loads(l) for l in open(f'{KB}/processed/findings_v2.jsonl',encoding='utf-8')]
ch=[r for r in rows if r['changed'] and r['final']['status']=='accepted']
for r in random.sample(ch,15):
    print(f"{r['clause_id']:<16} {r['original']['relationship']} → {r['final']['relationship']}"
          f"  ({r['original']['confidence']} → {r['final']['confidence']})")
    print(f"   {r['ruling'].get('reasoning','')[:200]}\n")

3.3.16-3.b       intersects_with → subset_of  (0.6 → 0.75)
   The SAMA clause specifically concerns gathering threat intelligence from diverse external sources, which is the core intent of PM-16's threat awareness/information-sharing program, whereas SI-5 is nar

3.3.17-3.a       subset_of → superset_of  (0.65 → 0.75)
   RA-5 addresses vulnerability scanning only for 'the system and hosted applications', while the SAMA clause's scope element explicitly covers 'all information assets', making SAMA the broader term with

3.4.2-3.b        intersects_with → subset_of  (0.55 → 0.65)
   The SAMA clause's requirement for cyber security function involvement is a specific instance of NIST's broader requirement for approval by defined personnel or roles in outsourcing decisions; NIST's a

3.3.3-3.a        subset_of → subset_of  (0.65 → 0.6)
   CM-8 requires a comprehensive, accurate inventory of system components, which more closely matches the narrower concept of a unified asset register than P

In [9]:
!cd {KB} && python debate.py run --band provisional

resuming: 264 already debated
nothing to do.


In [10]:
import json, random
random.seed(3)
rows=[json.loads(l) for l in open(f'{KB}/processed/findings.jsonl',encoding='utf-8')]
cl={c['clause_id']:c for c in (json.loads(l) for l in open(f'{KB}/processed/sama_clauses.jsonl',encoding='utf-8'))}
cand={json.loads(l)['clause_id']:json.loads(l) for l in open(f'{KB}/processed/candidates.jsonl',encoding='utf-8')}

gaps=[r for r in rows if not r['findings'] and cl[r['clause_id']]['clause_type']!='referential']
for r in random.sample(gaps,30):
    c=cl[r['clause_id']]
    print(f"{r['clause_id']:<18} [{c['clause_type']}]  {c['text'][:110]}")
    print(f"   عُرض: {[x['control_id'] for x in cand[r['clause_id']]['candidates'][:6]]}\n")

3.1.4-2.c.1        [substantive]  the cyber security governance;
   عُرض: ['SA-9', 'PL-8', 'SA-4', 'PM-23', 'SA-24', 'PS-9']

3.2.4-5.b          [substantive]  critical risks have been treated effectively;
   عُرض: ['SA-11', 'AU-6', 'AU-6(4)', 'SA-24', 'SA-11(4)', 'SA-15']

3.2.1.4-1.b        [substantive]  the selected and agreed cyber security controls are being implemented.
   عُرض: ['SI-4', 'SA-9', 'RA-3', 'SA-4', 'SI-5', 'SA-11']

3.1.3-3.a          [substantive]  considered as input for other corporate policies of the Member Organization (e.g., HR policy, finance policy a
   عُرض: ['SI-5', 'SI-4', 'PS-6', 'PS-1', 'SI-1', 'RA-1']

3.1.4-4.i.5.a      [substantive]  performing cyber security audits.
   عُرض: ['SA-9', 'SA-4', 'PS-9', 'SA-24', 'SA-8(22)', 'PM-29']

3.3.1-3.b          [substantive]  staff should receive cyber security awareness at the start and during their employment;
   عُرض: ['SA-15', 'PS-4', 'PS-4(1)', 'SA-8(27)', 'PM-12', 'PS-6(3)']

3.2.1-8.c          [substantiv

In [11]:
!cd {KB} && python app.py selftest

data ready: True 
  clauses 489  controls 1196 (1014 active)  findings rows 407
  accepted 186  verified 526
  clause gaps 148: below threshold=90, not tested=43, evidenced absence=15
  local specificity (superset_of): 20
  excluded as list items / referential: 158 of 407
  weakest subdomain: 3.3.8 — 3 no match, 4 for review, of 11  [below threshold]
  mapping opens on: 3.3.8-6.g
  control gaps: worst family CP 100%
  parameter gaps: hard 4 soft 45
  drift: {'SR': {'total': 27, 'matched': 1, 'title': 'Supply Chain Risk Management'}, 'PT': {'total': 21, 'matched': 0, 'title': 'Personally Identifiable Information Processing and Transparency'}, 'rev520': {'known': ['SA-15(13)', 'SI-2(7)', 'SA-24'], 'matched': ['SI-2(7)']}}
  clause choices 404
  detail rows for 3.1.1-1: 0
  kb rows 18
  itu factors 6 dimensions 13

## SANAD — evidence-anchored regulatory alignment

**SAMA Cyber Security Framework (2017)** against **NIST SP 800-53 Rev 5.2.0**,
with every claim anchored to a verbatim span o

In [12]:
import json
from collections import Counter
import sys; sys.path.insert(0, KB)
import app as A
d = A.Data()
g = A.clause_gaps(d)
sev = Counter(x['severity'] for x in g)

mappable = sum(1 for r in d.findings
               if d.clauses.get(r['clause_id'],{}).get('clause_type')!='referential'
               and not A.enumerative(d.clauses.get(r['clause_id'],{})))
acc = len({cid for cid,_ in d.accepted
           if not A.enumerative(d.clauses.get(cid,{}))})

print(f"المقام المصحَّح : {mappable}")
print(f"مقبول          : {acc}  ({100*acc/mappable:.1f}%)")
for k,v in sev.most_common():
    print(f"{k:<20}: {v}  ({100*v/mappable:.1f}%)")

# التغطية عند خفض العتبة
ver = len({cid for cid,_ in d.verified if not A.enumerative(d.clauses.get(cid,{}))})
print(f"\nمُتحقَّق (يشمل دون العتبة): {ver}  ({100*ver/mappable:.1f}%)")

المقام المصحَّح : 249
مقبول          : 101  (40.6%)
below threshold     : 90  (36.1%)
not tested          : 43  (17.3%)
evidenced absence   : 15  (6.0%)

مُتحقَّق (يشمل دون العتبة): 190  (76.3%)


In [13]:
import sys; sys.path.insert(0, KB)
import app as A
d = A.Data()

acc_all = {cid for cid,_ in d.accepted}
acc_map = {cid for cid,_ in d.accepted
           if not A.enumerative(d.clauses.get(cid,{}))
           and d.clauses.get(cid,{}).get('clause_type') != 'referential'}

print('بنود بحكم مقبول (خام)   :', len(acc_all))
print('بنود بحكم مقبول (مصحَّح):', len(acc_map))
print('الفارق                  :', len(acc_all - acc_map))
print()
for cid in sorted(acc_all - acc_map)[:10]:
    c = d.clauses.get(cid, {})
    f = next(f for x,f in d.accepted if x == cid)
    print(f"{cid:<18} {f['control_id']:<10} {f['confidence']}  ({len(c.get('text',''))} حرفاً)")
    print(f"   {c.get('text','')[:110]}\n")

بنود بحكم مقبول (خام)   : 158
بنود بحكم مقبول (مصحَّح): 101
الفارق                  : 57

3.1.1-3.b          PM-2       0.85  (42 حرفاً)
   Chief information security officer (CISO);

3.1.3-4.f.4        RA-3       0.85  (69 حرفاً)
   cyber security risk assessments are conducted for information assets;

3.1.4-1.a          PM-3       0.75  (64 حرفاً)
   ensuring that sufficient budget for cyber security is allocated;

3.1.4-2.c.5        RA-3       0.6  (39 حرفاً)
   cyber security risk management process;

3.1.5-2.e          SA-3       0.75  (62 حرفاً)
   responsibilities for cyber security are defined and allocated;

3.1.6-2.a          AT-2       0.85  (33 حرفاً)
   staff of the Member Organization;

3.1.6-5.b          AT-3       0.7  (56 حرفاً)
   the roles and responsibilities regarding cyber security;

3.1.7-1.a          AT-3       0.75  (34 حرفاً)
   key roles within the organization;

3.1.7-1.b          AT-3       0.75  (37 حرفاً)
   staff of the cyber security function;

3.1.7-1.

In [14]:
import re, pathlib
p = pathlib.Path(f'{KB}/app.py'); s = p.read_text(encoding='utf-8')

s = s.replace('''    txt = (c.get("text") or "").strip()
    return bool(
        c.get("deontic_inherited")
        and c.get("parent_id")
        and len(txt) < 70
        and txt.endswith((";", "."))
    )''',
'''    txt = (c.get("text") or "").strip()
    if not (c.get("deontic_inherited") and c.get("parent_id")
            and len(txt) < 70 and txt.endswith((";", "."))):
        return False
    # A requirement carries a verb; a list item is a noun phrase. Without this
    # test, "responsibilities are defined and allocated;" is discarded beside
    # "staff involved in risk assessments." — a requirement and a role name.
    return not re.search(
        r"\\b(is|are|was|were|be|been|shall|should|must|may|will|can)\\b"
        r"|\\b\\w+(?:ing|ed|es|s)\\b\\s+(?:for|to|by|with|in|on|that|the)\\b",
        txt, re.I)''')

if 'import re' not in s.split('\\n\\n')[0]:
    s = s.replace('import csv\\nimport json', 'import csv\\nimport json\\nimport re')
p.write_text(s, encoding='utf-8')
print('patched')

patched


In [15]:
!cd {KB} && python app.py selftest

data ready: True 
  clauses 489  controls 1196 (1014 active)  findings rows 407
  accepted 186  verified 526
Traceback (most recent call last):
  File "/content/drive/MyDrive/sanad-ai-readiness/kb/app.py", line 811, in <module>
    sys.exit(selftest() if "selftest" in sys.argv else launch("--no-share" not in sys.argv))
             ^^^^^^^^^^
  File "/content/drive/MyDrive/sanad-ai-readiness/kb/app.py", line 565, in selftest
    g = clause_gaps(d)
        ^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/sanad-ai-readiness/kb/app.py", line 184, in clause_gaps
    if c.get("clause_type") == "referential" or enumerative(c):
                                                ^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/sanad-ai-readiness/kb/app.py", line 145, in enumerative
    return not re.search(
               ^^
NameError: name 're' is not defined. Did you forget to import 're'?


In [16]:
import pathlib
p = pathlib.Path(f'{KB}/app.py'); s = p.read_text(encoding='utf-8')
if '\nimport re\n' not in s:
    s = s.replace('import json\n', 'import json\nimport re\n', 1)
    p.write_text(s, encoding='utf-8')
print([l for l in s.split('\n')[:35] if l.startswith('import')])

['import csv', 'import json', 'import re', 'import os', 'import sys']


In [17]:
!cd {KB} && python app.py selftest

data ready: True 
  clauses 489  controls 1196 (1014 active)  findings rows 407
  accepted 186  verified 526
  clause gaps 169: below threshold=99, not tested=53, evidenced absence=17
  local specificity (superset_of): 20
  excluded as list items / referential: 126 of 407
  weakest subdomain: 3.1.5 — 4 no match, 1 for review, of 7  [evidenced absence]
  mapping opens on: 3.3.8-6.g
  control gaps: worst family CP 100%
  parameter gaps: hard 4 soft 45
  drift: {'SR': {'total': 27, 'matched': 1, 'title': 'Supply Chain Risk Management'}, 'PT': {'total': 21, 'matched': 0, 'title': 'Personally Identifiable Information Processing and Transparency'}, 'rev520': {'known': ['SA-15(13)', 'SA-24', 'SI-2(7)'], 'matched': ['SI-2(7)']}}
  clause choices 404
  detail rows for 3.1.1-1: 0
  kb rows 18
  itu factors 6 dimensions 13

## SANAD — evidence-anchored regulatory alignment

**SAMA Cyber Security Framework (2017)** against **NIST SP 800-53 Rev 5.2.0**,
with every claim anchored to a verbatim span 

In [18]:
import pathlib
p = pathlib.Path(f'{KB}/app.py'); s = p.read_text(encoding='utf-8')

old = '''    covered = len({cid for cid, _ in d.accepted})'''
new = '''    covered = len({cid for cid, _ in d.accepted
                   if not enumerative(d.clauses.get(cid, {}))
                   and d.clauses.get(cid, {}).get("clause_type") != "referential"})'''
assert old in s, 'لم يُعثر على الموضع'
s = s.replace(old, new)
p.write_text(s, encoding='utf-8')
print('✓ البسط صار مصحَّحاً')

✓ البسط صار مصحَّحاً


In [19]:
import sys; sys.path.insert(0, KB)
import importlib, app as A
importlib.reload(A)
d = A.Data()

def ok(cid):
    c = d.clauses.get(cid, {})
    return not A.enumerative(c) and c.get('clause_type') != 'referential'

m   = sum(1 for r in d.findings if ok(r['clause_id']))
acc = len({cid for cid,_ in d.accepted  if ok(cid)})
ver = len({cid for cid,_ in d.verified  if ok(cid)})
from collections import Counter
sev = Counter(x['severity'] for x in A.clause_gaps(d))

print(f"المقام المصحَّح      : {m}")
print(f"مقبول (τ=0.60)      : {acc}  ({100*acc/m:.1f}%)")
print(f"متحقَّق (كل الأحكام) : {ver}  ({100*ver/m:.1f}%)")
for k in ('evidenced absence','not tested','below threshold'):
    print(f"{k:<20}: {sev[k]:>3}  ({100*sev[k]/m:.1f}%)")

المقام المصحَّح      : 281
مقبول (τ=0.60)      : 112  (39.9%)
متحقَّق (كل الأحكام) : 210  (74.7%)
evidenced absence   :  17  (6.0%)
not tested          :  53  (18.9%)
below threshold     :  99  (35.2%)


In [20]:
!cd {KB} && python precision.py sample -n 30

pool: 186 accepted findings (single-judge)
  confirmed     113
  provisional    73
  weak            0

sampled 30 -> processed/review.jsonl
seed 17; re-run with the same seed to reproduce the sample,
or with a different one for a second, independent reviewer.

Next:  python precision.py label


In [21]:
!cd {KB} && python precision.py label

30 left of 30. Labels: [c]orrect  [t] wrong type  [w]rong  [u]nsure  [s]kip  [q]uit

#1   3.3.3-3.a  ->  CM-8
claim: subset_of    (confidence hidden until scoring)

SAMA 3.3.3 Asset Management
  a unified register;

NIST CM-8 — System Component Inventory
  a. Develop and document an inventory of system components that:
1. Accurately reflects the system;
2. Includes all components within the system;
3. Does not include duplicate accounting of components or components assigned to any other system;
4. Is at the level of granularity deemed necessary for tracking and reporting; and
5. Includes the following information to achieve system component accountability: [information] ; and
b. Review and update the system component inventory [frequency].

quoted from SAMA: “a unified register;”
quoted from NIST: “Develop and document an inventory of system components that:”
stated reason   : CM-8 requires a system component inventory that could serve as a unified register, but is narrower in scope t

In [22]:
!cd {KB} && python precision.py score

PRECISION OF ACCEPTED MAPPINGS
labelled            : 30
excluded as unsure  : 0
decided             : 30

correct             : 22
wrong relationship  : 6
not a relationship  : 2

precision           : 73.3%   95% CI [55.6%, 85.8%]

by confidence band
  confirmed     13/18   72.2%  [49%, 88%]
  provisional    9/12   75.0%  [47%, 91%]

by relationship claimed
  equal                2/2
  intersects_with      5/5
  subset_of           14/22
  superset_of          1/1

the 8 that failed
  3.3.3-3.a          CM-8       subset_of        wrong
  3.3.14-4.j         SI-4(13)   subset_of        wrong_type
  3.3.5-4.f.4.a      AC-2       subset_of        wrong
  3.3.5-4.b.7        AC-2       subset_of        wrong_type
  3.3.5-4.e          IA-2(2)    subset_of        wrong_type
  3.3.7-4.h          CM-3       subset_of        wrong_type
  3.4.1-1            SA-4       subset_of        wrong_type
  3.3.13-4.b.2       SC-35      subset_of        wrong_type

Report as: precision 73% on a stratified

In [24]:
!cd {KB} && python precision.py sample -n 30 --debated --seed 23

pool: 212 accepted findings (debated)
  confirmed      96
  provisional   116
  weak            0

sampled 30 -> processed/review.jsonl
seed 23; re-run with the same seed to reproduce the sample,
or with a different one for a second, independent reviewer.

Next:  python precision.py label


In [25]:
%%shell
cd /content/drive/MyDrive/sanad-ai-readiness/kb && python precision.py label

30 left of 30. Labels: [c]orrect  [t] wrong type  [w]rong  [u]nsure  [s]kip  [q]uit

#1   3.3.4-3.d  ->  SA-8
claim: equal    (confidence hidden until scoring)

SAMA 3.3.4 Cyber Security Architecture
  design principles for developing cyber security controls and applying cyber security requirements (i.e., the security-by-design principle);

NIST SA-8 — Security and Privacy Engineering Principles
  Apply the following systems security and privacy engineering principles in the specification, design, development, implementation, and modification of the system and system components: [organization-defined systems security and privacy engineering principles].

quoted from SAMA: “design principles for developing cyber security controls and applying cyber security requirements (i.e., the security-by-design principle)”
quoted from NIST: “Apply the following systems security and privacy engineering principles in the specification, design, development, implementation, and modification of the syst

In [26]:
!cp {KB}/processed/review.jsonl {KB}/processed/review_debated.jsonl
!cd {KB} && python precision.py score

PRECISION OF ACCEPTED MAPPINGS
labelled            : 30
excluded as unsure  : 0
decided             : 30

correct             : 27
wrong relationship  : 0
not a relationship  : 3

precision           : 90.0%   95% CI [74.4%, 96.5%]

by confidence band
  confirmed     13/14   92.9%  [69%, 99%]
  provisional   14/16   87.5%  [64%, 97%]

by relationship claimed
  equal                1/1
  intersects_with      8/8
  subset_of           18/21

the 3 that failed
  3.1.3-1            RA-1       subset_of        wrong
  3.3.8-6.g          SI-2       subset_of        wrong
  3.1.3-1            PS-1       subset_of        wrong

Report as: precision 90% on a stratified sample of 30 accepted mappings, 95% CI [74%, 97%].
A sample this size supports a range, not a point estimate — quote the
interval, and state the sample size beside the figure.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
from google.colab import userdata
KB = '/content/drive/MyDrive/sanad-ai-readiness/kb'
os.environ['HF_HOME'] = '/content/drive/MyDrive/sanad-ai-readiness/.hf'

!pip install -q gradio sentence-transformers
!ls {KB}/processed/

Mounted at /content/drive
candidates.jsonl  findings_v2.jsonl	  routes.jsonl
chunks.jsonl	  never_shortlisted.json  sama_clauses.jsonl
controls.jsonl	  review_debated.jsonl	  text
findings.jsonl	  review.jsonl


In [4]:
!cd {KB} && python app.py selftest

data ready: True 
  clauses 489  controls 1196 (1014 active)  findings rows 407
  accepted 328  verified 512
  arbiter rulings applied: 252
  clause gaps 120: not tested=53, below threshold=50, evidenced absence=17
  local specificity (superset_of): 20
  excluded as list items / referential: 126 of 407
  weakest subdomain: 3.3.8 — 4 no match, 2 for review, of 11  [evidenced absence]
  mapping opens on: 3.3.8-6.g
  control gaps: worst family CP 100%
  parameter gaps: hard 6 soft 62
  drift: {'SR': {'total': 27, 'matched': 2, 'title': 'Supply Chain Risk Management'}, 'PT': {'total': 21, 'matched': 1, 'title': 'Personally Identifiable Information Processing and Transparency'}, 'rev520': {'known': ['SA-24', 'SA-15(13)', 'SI-2(7)'], 'matched': ['SI-2(7)']}}
  clause choices 404
  detail rows for 3.1.1-1: 0
  kb rows 18
  itu factors 6 dimensions 13

## SANAD — evidence-anchored regulatory alignment

**SAMA Cyber Security Framework (2017)** against **NIST SP 800-53 Rev 5.2.0**,
with every cl

In [3]:
import pathlib
p = pathlib.Path(f'{KB}/app.py'); s = p.read_text(encoding='utf-8')

if '\nimport re\n' not in s:
    s = s.replace('import json\n', 'import json\nimport re\n', 1)

old = '''    txt = (c.get("text") or "").strip()
    return bool(
        c.get("deontic_inherited")
        and c.get("parent_id")
        and len(txt) < 70
        and txt.endswith((";", "."))
    )'''
new = '''    txt = (c.get("text") or "").strip()
    if not (c.get("deontic_inherited") and c.get("parent_id")
            and len(txt) < 70 and txt.endswith((";", "."))):
        return False
    return not re.search(
        r"\\b(is|are|was|were|be|been|shall|should|must|may|will|can)\\b"
        r"|\\b\\w+(?:ing|ed|es|s)\\b\\s+(?:for|to|by|with|in|on|that|the)\\b",
        txt, re.I)'''
assert old in s, 'لم يُعثر على الموضع'
p.write_text(s.replace(old, new), encoding='utf-8')
print('✓ أُعيد الإصلاح')

✓ أُعيد الإصلاح


In [5]:
import sys; sys.path.insert(0, KB)
import importlib, app as A
importlib.reload(A)
d = A.Data()
ok = lambda cid: (not A.enumerative(d.clauses.get(cid,{}))
                  and d.clauses.get(cid,{}).get('clause_type')!='referential')
m   = sum(1 for r in d.findings if ok(r['clause_id']))
acc = len({cid for cid,_ in d.accepted if ok(cid)})
ver = len({cid for cid,_ in d.verified if ok(cid)})
print(f"مقبول : {acc}/{m} = {100*acc/m:.1f}%")
print(f"متحقَّق: {ver}/{m} = {100*ver/m:.1f}%")

مقبول : 161/281 = 57.3%
متحقَّق: 205/281 = 73.0%


In [6]:
!cd {KB} && python app.py

Traceback (most recent call last):
  File "/content/drive/MyDrive/sanad-ai-readiness/kb/app.py", line 869, in <module>
    sys.exit(selftest() if "selftest" in sys.argv else launch("--no-share" not in sys.argv))
                                                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/sanad-ai-readiness/kb/app.py", line 744, in launch
    f"### Broader than the reference — {len(sup)} findings\n"
                                            ^^^
NameError: name 'sup' is not defined. Did you mean: 'sum'?


In [7]:
import pathlib
p = pathlib.Path(f'{KB}/app.py'); s = p.read_text(encoding='utf-8')

old = '''    gaps = clause_gaps(d)
    sdgaps = gap_by_subdomain(d)
    cgaps = control_gaps(d)'''
new = '''    gaps = clause_gaps(d)
    sdgaps = gap_by_subdomain(d)
    cgaps = control_gaps(d)
    sup = local_specificity(d)'''

assert old in s, 'لم يُعثر على الموضع'
p.write_text(s.replace(old, new), encoding='utf-8')
print('✓ تم')

✓ تم


In [10]:
!cd {KB} && python app.py

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
Keyboard interruption in main thread... closing server.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 3367, in block_thread
    time.sleep(0.1)
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/drive/MyDrive/sanad-ai-readiness/kb/app.py", line 870, in <module>
    sys.exit(selftest() if "selftest" in sys.argv else launch("--no-share" not in sys.argv))
                                                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/sanad-ai-readiness/kb/app.py", line 863, in launch
    app.launch(share=share and not on_spaces,
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 3268, in launch
    self.block_thread()
 

In [9]:
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(7860)"))

https://7860-gpu-t4-s-kkb-usw1b2-12kk29bfq3j5s-b.us-west1-2.prod.colab.dev


In [11]:
import pathlib
p = pathlib.Path(f'{KB}/app.py'); s = p.read_text(encoding='utf-8')

old = '''    covered = len({cid for cid, _ in d.accepted})'''
new = '''    covered = len({cid for cid, _ in d.accepted
                   if not enumerative(d.clauses.get(cid, {}))
                   and d.clauses.get(cid, {}).get("clause_type") != "referential"})'''
assert old in s, 'لم يُعثر على الموضع'
p.write_text(s.replace(old, new), encoding='utf-8')
print('✓ تم')

✓ تم


In [13]:
import pathlib
p = pathlib.Path(f'{KB}/app.py'); s = p.read_text(encoding='utf-8')
n = 0

# ١ — import re
if '\nimport re\n' not in s:
    s = s.replace('import json\n', 'import json\nimport re\n', 1); n += 1

# ٢ — شرط الفعل في تصنيف عناصر التعداد
old = '''    txt = (c.get("text") or "").strip()
    return bool(
        c.get("deontic_inherited")
        and c.get("parent_id")
        and len(txt) < 70
        and txt.endswith((";", "."))
    )'''
if old in s:
    s = s.replace(old, '''    txt = (c.get("text") or "").strip()
    if not (c.get("deontic_inherited") and c.get("parent_id")
            and len(txt) < 70 and txt.endswith((";", "."))):
        return False
    return not re.search(
        r"\\b(is|are|was|were|be|been|shall|should|must|may|will|can)\\b"
        r"|\\b\\w+(?:ing|ed|es|s)\\b\\s+(?:for|to|by|with|in|on|that|the)\\b",
        txt, re.I)'''); n += 1

# ٣ — sup في launch
old = '''    cgaps = control_gaps(d)'''
if old in s and 'sup = local_specificity(d)\n    frows' not in s:
    s = s.replace(old, old + '\n    sup = local_specificity(d)', 1); n += 1

# ٤ — البسط المصحَّح
old = '''    covered = len({cid for cid, _ in d.accepted})'''
if old in s:
    s = s.replace(old, '''    covered = len({cid for cid, _ in d.accepted
                   if not enumerative(d.clauses.get(cid, {}))
                   and d.clauses.get(cid, {}).get("clause_type") != "referential"})'''); n += 1

p.write_text(s, encoding='utf-8')
print(f'طُبِّق {n} إصلاح')

import re as _re
print('import re :', '\nimport re\n' in s)
print('verb test :', 're.search' in s)
print('sup       :', s.count('sup = local_specificity(d)'))
print('covered   :', 'not enumerative(d.clauses.get(cid, {}))' in s)

طُبِّق 1 إصلاح
import re : True
verb test : True
sup       : 3
covered   : True


In [14]:
!cd {KB} && python app.py selftest

data ready: True 
  clauses 489  controls 1196 (1014 active)  findings rows 407
  accepted 328  verified 512
  arbiter rulings applied: 252
  clause gaps 120: not tested=53, below threshold=50, evidenced absence=17
  local specificity (superset_of): 20
  excluded as list items / referential: 126 of 407
  weakest subdomain: 3.3.8 — 4 no match, 2 for review, of 11  [evidenced absence]
  mapping opens on: 3.3.8-6.g
  control gaps: worst family CP 100%
  parameter gaps: hard 6 soft 62
  drift: {'SR': {'total': 27, 'matched': 2, 'title': 'Supply Chain Risk Management'}, 'PT': {'total': 21, 'matched': 1, 'title': 'Personally Identifiable Information Processing and Transparency'}, 'rev520': {'known': ['SI-2(7)', 'SA-15(13)', 'SA-24'], 'matched': ['SI-2(7)']}}
  clause choices 404
  detail rows for 3.1.1-1: 0
  kb rows 18
  itu factors 6 dimensions 13

## SANAD — evidence-anchored regulatory alignment

**SAMA Cyber Security Framework (2017)** against **NIST SP 800-53 Rev 5.2.0**,
with every cl

In [15]:
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(7860)"))

https://7860-gpu-t4-s-kkb-usw1b2-12kk29bfq3j5s-b.us-west1-2.prod.colab.dev


In [16]:
!cd {KB} && python app.py

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
Keyboard interruption in main thread... closing server.


In [17]:
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(7860)"))

https://7860-gpu-t4-s-kkb-usw1b2-12kk29bfq3j5s-b.us-west1-2.prod.colab.dev


In [18]:
!cd {KB} && python app.py

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
Keyboard interruption in main thread... closing server.


In [20]:
!pip install -q "gradio==4.44.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 12.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-client 1.3.0 which is incompatible.
langgraph-sdk 0.4.2 requires websockets<16,>=14, but you have websockets 12.0 which is incompatible.
google-adk 2.4.0 requires websockets<16,>=15.0.1, but you have websockets 12.0 which is incompatible.
google-genai 2.11.0 requires websockets<17.0,>=13.0.0, but you have websockets 12.0 which is incompatible.
langsmith 0.

In [19]:
import gradio;
print(gradio.__version__)


6.20.0


In [1]:
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(7860)"))

https://7860-gpu-t4-s-kkb-usw1b2-12kk29bfq3j5s-b.us-west1-2.prod.colab.dev


In [2]:
!cd {KB} && python app.py

/bin/bash: line 1: cd: {KB}: No such file or directory


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
KB = '/content/drive/MyDrive/sanad-ai-readiness/kb'
os.environ['HF_HOME'] = '/content/drive/MyDrive/sanad-ai-readiness/.hf'

!pip install -q "gradio==4.44.0" sentence-transformers
import gradio; print('gradio', gradio.__version__)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


ImportError: cannot import name 'HfFolder' from 'huggingface_hub' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/__init__.py)

In [4]:
!pip install -q "gradio==4.44.0" "huggingface_hub==0.25.2" sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
KB = '/content/drive/MyDrive/sanad-ai-readiness/kb'
os.environ['HF_HOME'] = '/content/drive/MyDrive/sanad-ai-readiness/.hf'
import gradio; print('gradio', gradio.__version__)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
gradio 4.44.0


In [4]:
!pip install -q "gradio==5.9.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.4/320.4 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 7.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-client 1.5.2 which is incompatible.
google-adk 2.4.0 requires starlette<2,>=1.0.1, but you have starlette 0.52.1 which is incompatible.
google-adk 2.4.0 requires websockets<16,>=15.0.1, but you have websockets 12.0 which is incompatible.
python-fasthtml 0.14.6 requires starlette>=1.0.1, but you have starlette 0.52.1 which is incompatible.


In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
KB = '/content/drive/MyDrive/sanad-ai-readiness/kb'
os.environ['HF_HOME'] = '/content/drive/MyDrive/sanad-ai-readiness/.hf'
import gradio; print('gradio', gradio.__version__)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
gradio 5.9.1


In [2]:
!cd {KB} && python app.py

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://bc60505c5aece6207b.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Keyboard interruption in main thread... closing server.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2869, in block_thread
    time.sleep(0.1)
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/drive/MyDrive/sanad-ai-readiness/kb/app.py", line 873, in <module>
    sys.exit(selftest() if "selftest" in sys.argv else launch("--no-share" not in sys.argv))
                                                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/sanad-ai-readiness/kb/app.py", line 866, in launch

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
KB = '/content/drive/MyDrive/sanad-ai-readiness/kb'
os.environ['HF_HOME'] = '/content/drive/MyDrive/sanad-ai-readiness/.hf'
!ls {KB}/manifest.csv && echo "---" && head -1 {KB}/manifest.csv

Mounted at /content/drive
/content/drive/MyDrive/sanad-ai-readiness/kb/manifest.csv
---
doc_id,tier,title_en,title_ar,issuing_body,country,doc_type,status,sector,year,version,url,url_accessed,http_status,format,language,is_official_translation,sha256,bytes,page_count,itu_factors,itu_dimensions,reference_pair,license_note,local_path,notes


In [2]:
import csv
rows = list(csv.DictReader(open(f'{KB}/manifest.csv', encoding='utf-8-sig')))
for r in rows:
    if r['tier'] == 'extended':
        print(f"  {r['doc_id']:<26} {r['title_en'][:50]}")

  SAU-CST-CLOUD-2023         Cloud Computing Regulatory Framework
  SAU-NDMO-DGPOL-2020        National Data Governance Interim Regulations
  SAU-SAMA-BCM-2017          Business Continuity Management Framework
  SAU-SAMA-CSFRB-2017        Cyber Security Framework (Rulebook HTML edition)
  SAU-SAMA-ITGF-2017         IT Governance Framework
  SAU-SDAIA-DEEPFAKE-2024    Deepfake Technology Risks and Recommendations
  SAU-SDAIA-DTR-2023         Regulation on Personal Data Transfer outside the K
  SAU-SDAIA-GENAIPRIV-2024   Generative AI Guidelines for the Private Sector an
  SAU-SDAIA-NSDAI-2020       National Strategy for Data and Artificial Intellig
  SAU-SDAIA-PDPLIR-2023      Implementing Regulation of the Personal Data Prote


In [3]:
import csv, pathlib, shutil
from collections import Counter

p = pathlib.Path(f'{KB}/manifest.csv')
shutil.copy(p, f'{KB}/manifest_full.csv')          # نسخة احتياطية

CITED = {
    'SAU-NDMO-DGPOL-2020',      # الوثيقة الحاكمة لعقدة C
    'SAU-SAMA-BCM-2017',        # أساس استبعاد عائلة CP
    'SAU-SDAIA-DTR-2023',       # يسند نتيجة توطين البيانات
    'SAU-SAMA-CSFRB-2017',      # نسخة HTML — تخصّ §7.3
    'SAU-SDAIA-PDPLIR-2023',    # اللائحة التنفيذية
}

rows = list(csv.DictReader(p.open(encoding='utf-8-sig')))
fn = list(rows[0].keys())

kept = []
for r in rows:
    if r['doc_id'] in CITED:
        r['tier'] = 'referenced'
        kept.append(r)
    elif r['tier'] in ('core', 'method'):
        kept.append(r)

with p.open('w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fn, lineterminator='\n')
    w.writeheader(); w.writerows(kept)

print(f'{len(rows)} → {len(kept)}')
print(Counter(r['tier'] for r in kept))
print('\nمحذوفة:', [r['doc_id'] for r in rows if r not in kept])

21 → 16
Counter({'core': 8, 'referenced': 5, 'method': 3})

محذوفة: ['SAU-CST-CLOUD-2023', 'SAU-SAMA-ITGF-2017', 'SAU-SDAIA-DEEPFAKE-2024', 'SAU-SDAIA-GENAIPRIV-2024', 'SAU-SDAIA-NSDAI-2020']


In [4]:
import csv
rows = list(csv.DictReader(open(f'{KB}/manifest.csv', encoding='utf-8-sig')))
for r in rows:
    if r['tier'] == 'referenced':
        print(f"{'✓' if r['url'] else '·'} {r['doc_id']}")
        print(f"   {r['title_en']}")
        print(f"   {r['url'] or '← يحتاج رابطاً'}\n")

· SAU-NDMO-DGPOL-2020
   National Data Governance Interim Regulations
   ← يحتاج رابطاً

· SAU-SAMA-BCM-2017
   Business Continuity Management Framework
   ← يحتاج رابطاً

✓ SAU-SAMA-CSFRB-2017
   Cyber Security Framework (Rulebook HTML edition)
   https://rulebook.sama.gov.sa/en/cyber-security-framework-2

· SAU-SDAIA-DTR-2023
   Regulation on Personal Data Transfer outside the Kingdom
   ← يحتاج رابطاً

· SAU-SDAIA-PDPLIR-2023
   Implementing Regulation of the Personal Data Protection Law
   ← يحتاج رابطاً



In [5]:
!cd {KB} && python fetch.py validate
!cd {KB} && python fetch.py verify

Validating 16 records with jsonschema

  line 10  SAU-NDMO-DGPOL-2020
      - 'referenced' is not one of ['core', 'extended', 'method']
  line 11  SAU-SAMA-BCM-2017
      - 'referenced' is not one of ['core', 'extended', 'method']
  line 12  SAU-SAMA-CSFRB-2017
      - 'referenced' is not one of ['core', 'extended', 'method']
  line 13  SAU-SDAIA-DTR-2023
      - 'referenced' is not one of ['core', 'extended', 'method']
  line 14  SAU-SDAIA-PDPLIR-2023
      - 'referenced' is not one of ['core', 'extended', 'method']

  NOTE: 6 record(s) have no url yet (0 of them CORE):
      -      INT-ITU-AIREADY-2025  AI Ready — Analysis Towards a Standardized Readiness Fr
      -      INT-ITU-Y3172-2019  Architectural framework for machine learning in future 
      -      SAU-NDMO-DGPOL-2020  National Data Governance Interim Regulations
      -      SAU-SAMA-BCM-2017  Business Continuity Management Framework
      -      SAU-SDAIA-DTR-2023  Regulation on Personal Data Transfer outside the Kingdo
 

In [6]:
import csv, pathlib

FIX = {
 'USA-NIST-AIRMF-2023': {
   'year': '2023',
   'notes': 'Reference target for the SDAIA AI-ethics pair. Published January 2023.'},
 'SAU-NCA-ECC-2024': {
   'issuing_body': 'National Cybersecurity Authority (NCA)',
   'notes': 'ECC-2:2024, current edition. Cross-sector baseline; bridges the '
            'finance-specific SAMA framework to national controls.'},
 'SAU-SAMA-CSF-2017': {
   'issuing_body': 'Saudi Central Bank (SAMA)'},
 'SAU-SAMA-BCM-2017': {
   'issuing_body': 'Saudi Central Bank (SAMA)',
   'notes': 'Published as an HTML rulebook page; no PDF release. Basis for the '
            'documented exclusion of the CP control family: SAMA CSF 1.3 '
            'delegates business continuity to this instrument.'},
 'SAU-SDAIA-PDPL-2023': {
   'notes': 'Royal Decree M/19, 2021; amended 2023 (V2, 23 April 2023). Arabic '
            'text authoritative; English is an official translation.'},
 'SAU-SDAIA-AIADOPT-2025': {
   'notes': 'Four maturity levels; enablers = data, technology, human '
            'capabilities, responsible use. Those four correspond to the ITU '
            'readiness factors — a national correspondence reported in the paper.'},
 'SAU-SDAIA-PDPLIR-2023': {
   'notes': 'Implementing regulation for the PDPL. Cited for privacy-by-design '
            'justification; not indexed.'},
}
LICENSE = ('Publicly available on the issuing body\'s site. Redistribution '
           'rights unconfirmed — raw copy kept local, not committed.')

p = pathlib.Path(f'{KB}/manifest.csv')
rows = list(csv.DictReader(p.open(encoding='utf-8-sig')))
fn = list(rows[0].keys())
for r in rows:
    r.update(FIX.get(r['doc_id'], {}))
    if not (r.get('license_note') or '').strip():
        r['license_note'] = LICENSE
with p.open('w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fn, lineterminator='\n')
    w.writeheader(); w.writerows(rows)

import re
bad = [r['doc_id'] for r in rows
       if re.search(r'\b(VERIFY|Confirm|HIGH PRIORITY|TODO|check )', r['notes'] or '', re.I)]
print('ملاحظات ما زالت تعليمات:', bad or '✓ لا شيء')
print('بلا license_note:', [r['doc_id'] for r in rows if not r['license_note']] or '✓')

ملاحظات ما زالت تعليمات: ✓ لا شيء
بلا license_note: ✓


In [7]:
import csv, pathlib
p = pathlib.Path(f'{KB}/manifest.csv')
rows = list(csv.DictReader(p.open(encoding='utf-8-sig')))
fn = list(rows[0].keys())
for r in rows:
    if r['doc_id'] == 'SAU-SDAIA-PDPL-2023':
        r['version'] = 'M/19 (2021), amended by M/148 (2023)'
        r['year'] = '2023'
        r['notes'] = ('Issued by Royal Decree M/19 of 09/02/1443 AH '
                      '(16/09/2021); amended by Royal Decree M/148 of '
                      '05/09/1444 AH (27/03/2023). English edition V2, '
                      '23 April 2023. Arabic text authoritative.')
with p.open('w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fn, lineterminator='\n')
    w.writeheader(); w.writerows(rows)
print('✓')

✓


In [8]:
import pathlib
p = pathlib.Path(f'{KB}/app.py'); s = p.read_text(encoding='utf-8')

old = '''            m.get("doc_id", ""), m.get("tier", ""), m.get("title_en", "")[:60],
            m.get("issuing_body", ""), m.get("status", ""), m.get("year", ""),
            "yes" if m.get("sha256") else "no", m.get("url", "")[:70],'''
new = '''            m.get("doc_id", ""), m.get("tier", ""), m.get("title_en", "")[:60],
            m.get("issuing_body", ""), m.get("status", ""), m.get("year", ""),
            "yes" if m.get("sha256") else "no",
            m.get("reference_pair", "") or "\\u2014", m.get("url", "")[:70],'''
assert old in s, 'لم يُعثر على الموضع'
s = s.replace(old, new)

s = s.replace(
'''KB_COLS = ["doc_id", "tier", "title", "issuer", "legal status", "year", "held", "url"]''',
'''KB_COLS = ["doc_id", "tier", "title", "issuer", "legal status", "year", "held",
           "analysed against", "url"]''')

s = s.replace('''column_widths=["20%", "8%", "26%", "9%", "10%",
                                            "7%", "6%", "14%"],''',
              '''column_widths=["17%", "8%", "22%", "9%", "9%",
                                            "6%", "5%", "14%", "10%"],''')

p.write_text(s, encoding='utf-8')
print('✓ أُضيف عمود reference_pair')

✓ أُضيف عمود reference_pair


In [9]:
import csv, pathlib
p = pathlib.Path(f'{KB}/manifest.csv')
rows = list(csv.DictReader(p.open(encoding='utf-8-sig')))
fn = list(rows[0].keys())
for r in rows:
    if r['doc_id'] == 'SAU-NCA-ECC-2024':
        r['notes'] = (r['notes'].rstrip('. ') +
            '. Pair configured against SP 800-53 but not executed: the '
            'demonstrated pair is finance-sector (SAMA), and NCA controls are '
            'cross-sector.')
with p.open('w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fn, lineterminator='\n')
    w.writeheader(); w.writerows(rows)
print('✓')

✓


In [10]:
import csv, pathlib
from datetime import date
TODAY = date.today().isoformat()

p = pathlib.Path(f'{KB}/manifest.csv')
rows = list(csv.DictReader(p.open(encoding='utf-8-sig')))
fn = list(rows[0].keys())

n = 0
for r in rows:
    if r['tier'] in ('referenced', 'method') and not (r['url_accessed'] or '').strip():
        r['url_accessed'] = TODAY
        note = r['notes'].rstrip('. ')
        r['notes'] = (note + '. ' if note else '') + \
                     f'URL verified {TODAY}; cited, not retrieved.'
        n += 1

with p.open('w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fn, lineterminator='\n')
    w.writeheader(); w.writerows(rows)
print(f'✓ {n} صفوف')
print('بلا تاريخ:', [r['doc_id'] for r in rows if not r['url_accessed']] or 'لا شيء')

✓ 7 صفوف
بلا تاريخ: لا شيء


In [11]:
import csv, json, pathlib
from datetime import date
TODAY = date.today().isoformat()

# ١ — المخطط
sp = pathlib.Path(f'{KB}/manifest.schema.json')
sch = json.loads(sp.read_text(encoding='utf-8'))
sch['properties']['tier']['enum'] = ['core', 'referenced', 'method']
sch['properties']['tier']['description'] = (
    'core = ingested and analysed; referenced = cited in the report with a '
    'verified URL, not retrieved or indexed; method = methodological reference.')
sp.write_text(json.dumps(sch, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')

# ٢ — البيان
p = pathlib.Path(f'{KB}/manifest.csv')
rows = list(csv.DictReader(p.open(encoding='utf-8-sig')))
fn = list(rows[0].keys())

for r in rows:
    d = r['doc_id']

    if d == 'SAU-SAMA-CSF-2017':
        r['format'] = 'pdf'
        r['notes'] = (
            f'4 domains, 32 subdomains, 493 parsed clauses (407 leaf). Contains '
            f'no standards-mapping appendix. SAMA publishes no stable direct '
            f'file URL: the recorded url is the rulebook page from which the '
            f'PDF downloads. The sha256 and page_count describe that downloaded '
            f'PDF, not the page response, so `verify --upstream` will report a '
            f'false drift by design. URL verified {TODAY}.')

    elif d == 'USA-NIST-SP80053-2020':
        r['year'] = '2025'

    elif d == 'SAU-SDAIA-AIETHICS-2025':
        r['doc_type'] = 'guidelines'

    elif d == 'INT-ITU-Y3172-2019':
        r['notes'] = (
            f'Recommendation release page; the PDF downloads from the page '
            f'without a stable file URL. Approved 2019-06-22, in force. '
            f'Methodological reference — cited, not indexed. '
            f'URL verified {TODAY}.')

    elif d in ('SAU-SDAIA-DTR-2023', 'SAU-SDAIA-PDPLIR-2023'):
        note = r['notes'].rstrip('. ')
        r['notes'] = (note + '. ' if note else '') + (
            f'Portal URL contains session-like path segments and may not '
            f'persist; verified {TODAY}.')

    # اتساق أسماء الجهات
    r['issuing_body'] = {
        'SDAIA': 'Saudi Data and AI Authority (SDAIA)',
        'NCA':   'National Cybersecurity Authority (NCA)',
        'SAMA':  'Saudi Central Bank (SAMA)',
        'NIST':  'National Institute of Standards and Technology (NIST)',
        'ITU':   'International Telecommunication Union (ITU)',
        'ITU-T': 'International Telecommunication Union (ITU-T)',
    }.get(r['issuing_body'].strip(), r['issuing_body'])

with p.open('w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fn, lineterminator='\n')
    w.writeheader(); w.writerows(rows)

print('✓', len(rows), 'صفاً')
print(sorted({r['issuing_body'] for r in rows}))

✓ 15 صفاً
['International Telecommunication Union (ITU)', 'International Telecommunication Union (ITU-T)', 'National Cybersecurity Authority (NCA)', 'National Data Management Office (NDMO), SDAIA', 'National Institute of Standards and Technology (NIST)', 'Saudi Central Bank (SAMA)', 'Saudi Data and AI Authority (SDAIA)']


In [13]:
import csv, pathlib
from datetime import date
TODAY = date.today().isoformat()

p = pathlib.Path(f'{KB}/manifest.csv')
rows = list(csv.DictReader(p.open(encoding='utf-8-sig')))
fn = list(rows[0].keys())
for r in rows:
    if r['doc_id'] in ('SAU-SDAIA-DTR-2023', 'SAU-SDAIA-PDPLIR-2023'):
        note = r['notes'].rstrip('. ')
        r['notes'] = (note + '. ' if note else '') + (
            f'Published only through the SDAIA data-governance portal, whose '
            f'URLs carry session-like path segments and may not persist; '
            f'no stable short link found. Verified {TODAY}, snapshot retained.')
with p.open('w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fn, lineterminator='\n')
    w.writeheader(); w.writerows(rows)
print('✓')

✓


In [14]:
import csv, pathlib
TODAY = '2026-08-08'

p = pathlib.Path(f'{KB}/manifest.csv')
rows = list(csv.DictReader(p.open(encoding='utf-8-sig')))
fn = list(rows[0].keys())

blank = {k: '' for k in fn}
blank.update({
    'doc_id': 'SAU-SAMA-CSFRB-2017',
    'tier': 'referenced',
    'title_en': 'Cyber Security Framework — SAMA Rulebook web edition',
    'title_ar': 'إطار الأمن السيبراني — النسخة الإلكترونية',
    'issuing_body': 'Saudi Central Bank (SAMA)',
    'country': 'SAU',
    'doc_type': 'framework',
    'status': 'binding',
    'sector': 'finance',
    'year': '2017',
    'version': '1.0',
    'url': 'https://rulebook.sama.gov.sa/en/cyber-security-framework-3',
    'url_accessed': TODAY,
    'format': 'html',
    'language': 'en',
    'is_official_translation': 'TRUE',
    'itu_factors': 'standards',
    'itu_dimensions': '13',
    'license_note': ("Publicly available on the issuing body's site. "
                     "Redistribution rights unconfirmed."),
    'notes': ('Second publication channel for the same instrument as '
              'SAU-SAMA-CSF-2017; cited as evidence on publication format, not '
              'analysed. Neither channel carries per-control identifiers or '
              f'structured fields. URL verified {TODAY}.'),
})

if not any(r['doc_id'] == blank['doc_id'] for r in rows):
    rows.append(blank)

order = {'core': 0, 'referenced': 1, 'method': 2}
rows.sort(key=lambda r: (order.get(r['tier'], 9), r['doc_id']))

with p.open('w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fn, lineterminator='\n')
    w.writeheader(); w.writerows(rows)

from collections import Counter
print(f'✓ {len(rows)} صفاً'); print(Counter(r['tier'] for r in rows))

✓ 16 صفاً
Counter({'core': 8, 'referenced': 5, 'method': 3})


In [15]:
import csv, json, pathlib
TODAY = '2026-08-08'

# المخطط
sp = pathlib.Path(f'{KB}/manifest.schema.json')
sch = json.loads(sp.read_text(encoding='utf-8'))
sch['properties']['tier']['enum'] = ['core', 'referenced', 'method']
sch['properties']['tier']['description'] = (
    'core = ingested and analysed; referenced = cited in the report with a '
    'verified URL, not retrieved or indexed; method = methodological reference.')
sch['properties']['format']['description'] = (
    'Detected on fetch; for manually retrieved copies, the format of the held '
    'file rather than the URL response (see notes).')
sp.write_text(json.dumps(sch, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')

# الملاحظات المكرّرة وتاريخ ساما
p = pathlib.Path(f'{KB}/manifest.csv')
rows = list(csv.DictReader(p.open(encoding='utf-8-sig')))
fn = list(rows[0].keys())

for r in rows:
    d = r['doc_id']
    if d in ('SAU-SDAIA-DTR-2023', 'SAU-SDAIA-PDPLIR-2023'):
        base = ('Relevant to the SAMA data-residency provision in 3.4.3. '
                if d.endswith('DTR-2023') else
                'Implementing regulation for the PDPL; cited for '
                'privacy-by-design justification. ')
        r['notes'] = base + (
            'Published only through the SDAIA data-governance portal, whose '
            'URLs carry session-like path segments and may not persist; no '
            f'stable short link found. Cited, not retrieved. URL verified {TODAY}.')
    elif d == 'SAU-SAMA-CSF-2017':
        r['url_accessed'] = TODAY
        r['notes'] = (
            '4 domains, 32 subdomains, 493 parsed clauses (407 leaf). Contains '
            'no standards-mapping appendix. SAMA publishes no stable direct file '
            'URL: the recorded url is the rulebook page from which the PDF '
            'downloads. The sha256 and page_count describe the PDF held locally '
            '(downloaded 2026-07-30), not the page response, so '
            f'`verify --upstream` reports a false drift by design. Page URL '
            f'verified {TODAY}.')

with p.open('w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fn, lineterminator='\n')
    w.writeheader(); w.writerows(rows)

print(f'✓ {len(rows)} صفاً')
print('لقطة مذكورة:', [r['doc_id'] for r in rows if 'snapshot' in r['notes']] or 'لا شيء ✓')

✓ 16 صفاً
لقطة مذكورة: لا شيء ✓


In [16]:
!cd {KB} && python fetch.py validate
!cd {KB} && python app.py selftest | head -14

Validating 16 records with jsonschema


PASS: 0 invalid record(s).
data ready: True 
  clauses 489  controls 1196 (1014 active)  findings rows 407
  accepted 328  verified 512
  arbiter rulings applied: 252
  clause gaps 120: not tested=53, below threshold=50, evidenced absence=17
  local specificity (superset_of): 20
  excluded as list items / referential: 126 of 407
  weakest subdomain: 3.3.8 — 4 no match, 2 for review, of 11  [evidenced absence]
  mapping opens on: 3.3.8-6.g
  control gaps: worst family CP 100%
  parameter gaps: hard 6 soft 62
  drift: {'SR': {'total': 27, 'matched': 2, 'title': 'Supply Chain Risk Management'}, 'PT': {'total': 21, 'matched': 1, 'title': 'Personally Identifiable Information Processing and Transparency'}, 'rev520': {'known': ['SA-15(13)', 'SA-24', 'SI-2(7)'], 'matched': ['SI-2(7)']}}
  clause choices 404
  detail rows for 3.1.1-1: 0


In [17]:
!cd {KB} && python app.py selftest | tail -8

**SAMA Cyber Security Framework (2017)** against **NIST SP 800-53 Rev 5.2.0**,
with every claim anchored to a verbatim span of both sources.

| | |
|---|---|
| clauses analysed | **281** mappable, 126 excluded as list items or referential |
| judgments | **541** — 328 accepted, 184 for review, 29 rejected at the gate |
| debated | 252 argued by th


In [18]:
!cd {KB} && python app.py selftest 2>&1 | grep -E "kb rows|itu factors|dimensions"

  kb rows 13
  itu factors 6 dimensions 13


In [19]:
import csv, pathlib
from collections import Counter

p = pathlib.Path(f'{KB}/manifest.csv')
rows = list(csv.DictReader(p.open(encoding='utf-8-sig')))
fn = list(rows[0].keys())
kept = [r for r in rows if r['doc_id'] != 'SAU-SAMA-CSFRB-2017']

with p.open('w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fn, lineterminator='\n')
    w.writeheader(); w.writerows(kept)

print(f'{len(rows)} → {len(kept)}')
print(Counter(r['tier'] for r in kept))

16 → 15
Counter({'core': 8, 'referenced': 4, 'method': 3})


In [20]:
import csv, pathlib

p = pathlib.Path(f'{KB}/manifest.csv')
rows = list(csv.DictReader(p.open(encoding='utf-8-sig')))
fn = list(rows[0].keys())

for r in rows:
    if r['doc_id'] == 'USA-NIST-SP80053-2020':
        r['notes'] = ('OSCAL content tag v1.5.0 (13 May 2026). doc_id year = '
                      'revision lineage (Rev. 5, 2020); year field = release '
                      'held (5.2.0, 2025).')

with p.open('w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fn, lineterminator='\n')
    w.writeheader(); w.writerows(rows)

bad = [r['doc_id'] for r in rows if '\n' in (r['notes'] or '')]
print('✓ استُبدلت')
print('ملاحظات بأسطر جديدة:', bad or 'لا شيء ✓')

✓ استُبدلت
ملاحظات بأسطر جديدة: لا شيء ✓


In [21]:
import csv, json
rows = list(csv.DictReader(open(f'{KB}/manifest.csv', encoding='utf-8-sig')))
from collections import Counter
print('الصفوف:', len(rows), Counter(r['tier'] for r in rows))
print('CSFRB موجود؟', any(r['doc_id']=='SAU-SAMA-CSFRB-2017' for r in rows))

s = json.load(open(f'{KB}/manifest.schema.json', encoding='utf-8'))
print('\ntier enum   :', s['properties']['tier']['enum'])
print('tier desc   :', s['properties']['tier'].get('description','‹فارغ›')[:90])
print('format desc :', s['properties']['format'].get('description','‹فارغ›')[:90])

الصفوف: 15 Counter({'core': 8, 'referenced': 4, 'method': 3})
CSFRB موجود؟ False

tier enum   : ['core', 'referenced', 'method']
tier desc   : core = ingested and analysed; referenced = cited in the report with a verified URL, not re
format desc : Detected on fetch; for manually retrieved copies, the format of the held file rather than 


In [22]:
!cd {KB} && python app.py selftest 2>&1 | grep -E "kb rows|itu factors"

  kb rows 12
  itu factors 6 dimensions 13


In [23]:
import csv
from collections import Counter
rows = [r for r in csv.DictReader(open(f'{KB}/manifest.csv', encoding='utf-8-sig'))
        if r['tier'] != 'method']
fc, dc = Counter(), Counter()
for r in rows:
    for f in (r['itu_factors'] or '').split(';'):
        if f.strip(): fc[f.strip()] += 1
    for d in (r['itu_dimensions'] or '').split(';'):
        if d.strip(): dc[d.strip()] += 1
print('FACTORS:', dict(fc))
print('DIMENSIONS:', {k: dc[k] for k in sorted(dc, key=int)})

FACTORS: {'standards': 7, 'data': 6, 'deployment_support': 2, 'research': 2}
DIMENSIONS: {'1': 1, '4': 4, '5': 2, '7': 3, '10': 10, '13': 3}


In [24]:
!cd {KB} && python appendix.py > appendix.tex && head -6 appendix.tex && wc -l appendix.tex

% Generated by appendix.py from manifest.csv — do not edit by hand.
\clearpage

\section*{Appendix A — References}
\addcontentsline{toc}{section}{Appendix A — References}

121 appendix.tex


In [25]:
import pathlib
p = pathlib.Path(f'{KB}/appendix.py'); s = p.read_text(encoding='utf-8')
old = '''    if r.get("title_ar"):
        bits.append(f"({tex(r['title_ar'])}).")
'''
assert old in s
p.write_text(s.replace(old, ''), encoding='utf-8')
print('✓')

✓


In [26]:
!cd {KB} && python appendix.py > appendix.tex && echo "أُعيد التوليد"

أُعيد التوليد


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
KB = '/content/drive/MyDrive/sanad-ai-readiness/kb'
os.environ['HF_HOME'] = '/content/drive/MyDrive/sanad-ai-readiness/.hf'
!ls {KB}/appendix.py {KB}/manifest.csv {KB}/processed/findings.jsonl

Mounted at /content/drive
/content/drive/MyDrive/sanad-ai-readiness/kb/appendix.py
/content/drive/MyDrive/sanad-ai-readiness/kb/manifest.csv
/content/drive/MyDrive/sanad-ai-readiness/kb/processed/findings.jsonl


In [2]:
!cd {KB} && python appendix.py > appendix.tex && grep -c "" {KB}/appendix.tex

220


In [3]:
import csv, pathlib

DTR = ('https://dgp.sdaia.gov.sa/wps/wcm/connect/e5bbede0-1119-4f70-b4ef-f043ce58d780/'
       'Regulation+on+Personal+Data+Transfer+Outside+the+Kingdom..pdf'
       '?MOD=AJPERES&CONVERT_TO=url&CACHEID=ROOTWORKSPACE-'
       'e5bbede0-1119-4f70-b4ef-f043ce58d780-p6OMj1M')

PDPLIR = ('https://dgp.sdaia.gov.sa/wps/wcm/connect/2a9a4744-a2ae-444f-bad2-6dacde843637/'
          'The+impelementing+regulation+of+the+personal+data+protection+law.pdf'
          '?MOD=AJPERES&CONVERT_TO=url&CACHEID=ROOTWORKSPACE-'
          '2a9a4744-a2ae-444f-bad2-6dacde843637-pSK-Oie')

EDITS = {
 'SAU-SDAIA-DTR-2023': {
   'url': DTR, 'url_accessed': '2026-08-09', 'format': 'pdf',
   'year': '2024', 'version': '2.0',
   'notes': ('Direct WCM file link (content-UUID based). doc_id year = original '
             'issuance lineage (2023); year field = version held (2.0, 2024). '
             'Cited in this report; not retrieved or indexed.')},
 'SAU-SDAIA-PDPLIR-2023': {
   'url': PDPLIR, 'url_accessed': '2026-08-09', 'format': 'pdf',
   'notes': ('Direct WCM file link (content-UUID based). Held copy carries no '
             'version marker or issue date on the document itself; year = '
             'original issuance (2023); text reflects the M/148 amendment. '
             'Cited in this report; not retrieved or indexed.')},
}

p = pathlib.Path(f'{KB}/manifest.csv')
rows = list(csv.DictReader(p.open(encoding='utf-8-sig')))
fn = list(rows[0].keys())

found = {r['doc_id'] for r in rows} & EDITS.keys()
assert found == EDITS.keys(), f'مفقود: {EDITS.keys() - found}'

for r in rows:
    r.update(EDITS.get(r['doc_id'], {}))

with p.open('w', encoding='utf-8-sig', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fn, lineterminator='\n')
    w.writeheader(); w.writerows(rows)

for r in rows:
    if r['doc_id'] in EDITS:
        print(f"{r['doc_id']}  year={r['year']}  ver={r['version'] or '—'}  "
              f"fmt={r['format']}  url={len(r['url'])} حرفاً")

SAU-SDAIA-DTR-2023  year=2024  ver=2.0  fmt=pdf  url=233 حرفاً
SAU-SDAIA-PDPLIR-2023  year=2023  ver=—  fmt=pdf  url=240 حرفاً


In [4]:
from google.colab import drive
drive.mount('/content/drive')
import os
KB = '/content/drive/MyDrive/sanad-ai-readiness/kb'
os.environ['HF_HOME'] = '/content/drive/MyDrive/sanad-ai-readiness/.hf'
!pip install -q "gradio==5.9.1"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.4/320.4 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 122.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.2/170.2 kB 18.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
python-fasthtml 0.14.6 requires starlette>=1.0.1, but you have starlette 0.52.1 which is incompatible.
google-adk 2.4.0 requires starlette<2,>=1.0.1, but you have starlette 0.52.1 which is incompatible.
google-adk 2.4.0 requires websockets<16,>=15.0.1, but you have websockets 14.2 which is incomp

In [1]:
import pathlib
p = pathlib.Path(f'{KB}/app.py'); s = p.read_text(encoding='utf-8')

old = '''            m.get("doc_id", ""), m.get("tier", ""), m.get("title_en", "")[:60],
            m.get("issuing_body", ""), m.get("status", ""), m.get("year", ""),
            "yes" if m.get("sha256") else "no", m.get("url", "")[:70],'''
new = '''            m.get("doc_id", ""), m.get("tier", ""), m.get("title_en", "")[:60],
            m.get("issuing_body", ""), m.get("status", ""), m.get("year", ""),
            "yes" if m.get("sha256") else "no",
            m.get("reference_pair", "") or "\\u2014", m.get("url", "")[:70],'''
assert old in s, 'لم يُعثر — قد يكون مُطبَّقاً'
s = s.replace(old, new)
s = s.replace(
'''KB_COLS = ["doc_id", "tier", "title", "issuer", "legal status", "year", "held", "url"]''',
'''KB_COLS = ["doc_id", "tier", "title", "issuer", "legal status", "year", "held",
           "analysed against", "url"]''')
s = s.replace('''column_widths=["20%", "8%", "26%", "9%", "10%",
                                            "7%", "6%", "14%"],''',
              '''column_widths=["17%", "8%", "22%", "9%", "9%",
                                            "6%", "5%", "14%", "10%"],''')
p.write_text(s, encoding='utf-8'); print('✓')

NameError: name 'KB' is not defined

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
KB = '/content/drive/MyDrive/sanad-ai-readiness/kb'
os.environ['HF_HOME'] = '/content/drive/MyDrive/sanad-ai-readiness/.hf'
import gradio; print('gradio', gradio.__version__, '| KB', KB)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
gradio 5.9.1 | KB /content/drive/MyDrive/sanad-ai-readiness/kb


In [3]:
import pathlib
p = pathlib.Path(f'{KB}/app.py'); s = p.read_text(encoding='utf-8')

old = '''            m.get("doc_id", ""), m.get("tier", ""), m.get("title_en", "")[:60],
            m.get("issuing_body", ""), m.get("status", ""), m.get("year", ""),
            "yes" if m.get("sha256") else "no", m.get("url", "")[:70],'''
new = '''            m.get("doc_id", ""), m.get("tier", ""), m.get("title_en", "")[:60],
            m.get("issuing_body", ""), m.get("status", ""), m.get("year", ""),
            "yes" if m.get("sha256") else "no",
            m.get("reference_pair", "") or "\\u2014", m.get("url", "")[:70],'''
assert old in s, 'لم يُعثر — قد يكون مُطبَّقاً'
s = s.replace(old, new)
s = s.replace(
'''KB_COLS = ["doc_id", "tier", "title", "issuer", "legal status", "year", "held", "url"]''',
'''KB_COLS = ["doc_id", "tier", "title", "issuer", "legal status", "year", "held",
           "analysed against", "url"]''')
s = s.replace('''column_widths=["20%", "8%", "26%", "9%", "10%",
                                            "7%", "6%", "14%"],''',
              '''column_widths=["17%", "8%", "22%", "9%", "9%",
                                            "6%", "5%", "14%", "10%"],''')
p.write_text(s, encoding='utf-8'); print('✓')

AssertionError: لم يُعثر — قد يكون مُطبَّقاً

In [4]:
import re, pathlib
s = pathlib.Path(f'{KB}/app.py').read_text(encoding='utf-8')

print('العمود مُطبَّق؟', 'analysed against' in s)
print()
print('--- KB_COLS ---')
m = re.search(r'KB_COLS = \[.*?\]', s, re.S)
print(m.group() if m else 'لم يوجد')
print()
print('--- kb_table ---')
m = re.search(r'def kb_table.*?return rows', s, re.S)
print(m.group() if m else 'لم يوجد')

العمود مُطبَّق؟ True

--- KB_COLS ---
KB_COLS = ["doc_id", "tier", "title", "issuer", "legal status", "year", "held",
           "analysed against", "url"]

--- kb_table ---
def kb_table(d: Data) -> List[List[str]]:
    rows = []
    for m in d.manifest:
        if m.get("tier") == "method":
            continue
        rows.append([
            m.get("doc_id", ""), m.get("tier", ""), m.get("title_en", "")[:60],
            m.get("issuing_body", ""), m.get("status", ""), m.get("year", ""),
            "yes" if m.get("sha256") else "no",
            m.get("reference_pair", "") or "\u2014", m.get("url", "")[:70],
        ])
    return rows


In [5]:
import re, pathlib
s = pathlib.Path(f'{KB}/app.py').read_text(encoding='utf-8')
m = re.search(r'"Every document.*?\)', s, re.S)
print(repr(m.group()) if m else 'لم توجد')

'"Every document is public, fetched from its issuing body, "\n                    "and pinned by SHA-256 so a finding can be traced to the "\n                    "exact version that produced it."\n                )'


In [6]:
import pathlib
p = pathlib.Path(f'{KB}/app.py'); s = p.read_text(encoding='utf-8')

old = ('"Every document is public, fetched from its issuing body, "\n'
       '                    "and pinned by SHA-256 so a finding can be traced to the "\n'
       '                    "exact version that produced it."')

new = ('"Every document public; core documents fetched from their "\n'
       '                    "issuing bodies and pinned by SHA-256, so a finding traces "\n'
       '                    "to the exact version that produced it. Referenced rows "\n'
       '                    "are cited with a verified URL but not retrieved. The SAMA "\n'
       '                    "framework was downloaded by hand from its rulebook page: "\n'
       '                    "SAMA publishes no stable direct file URL."')

assert old in s, 'لم يُعثر'
p.write_text(s.replace(old, new), encoding='utf-8')
print('✓ حُدّثت الجملة')

✓ حُدّثت الجملة


In [7]:
!cd {KB} && python app.py selftest 2>&1 | head -14

data ready: True 
  clauses 489  controls 1196 (1014 active)  findings rows 407
  accepted 328  verified 512
  arbiter rulings applied: 252
  clause gaps 120: not tested=53, below threshold=50, evidenced absence=17
  local specificity (superset_of): 20
  excluded as list items / referential: 126 of 407
  weakest subdomain: 3.3.8 — 4 no match, 2 for review, of 11  [evidenced absence]
  mapping opens on: 3.3.8-6.g
  control gaps: worst family CP 100%
  parameter gaps: hard 6 soft 62
  drift: {'SR': {'total': 27, 'matched': 2, 'title': 'Supply Chain Risk Management'}, 'PT': {'total': 21, 'matched': 1, 'title': 'Personally Identifiable Information Processing and Transparency'}, 'rev520': {'known': ['SA-15(13)', 'SA-24', 'SI-2(7)'], 'matched': ['SI-2(7)']}}
  clause choices 404
  detail rows for 3.1.1-1: 0


In [8]:
!cd {KB} && python appendix.py > appendix.tex && echo "✓ أُعيد التوليد"

✓ أُعيد التوليد


In [12]:
!cd {KB} && python app.py

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://a633e0b2de64820f1c.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Keyboard interruption in main thread... closing server.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2869, in block_thread
    time.sleep(0.1)
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/drive/MyDrive/sanad-ai-readiness/kb/app.py", line 883, in <module>
    sys.exit(selftest() if "selftest" in sys.argv else launch("--no-share" not in sys.argv))
                                                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/sanad-ai-readiness/kb/app.py", line 876, in launch

In [10]:
import requests
URLS = {
 'DTR': 'https://dgp.sdaia.gov.sa/wps/wcm/connect/e5bbede0-1119-4f70-b4ef-f043ce58d780/Regulation+on+Personal+Data+Transfer+Outside+the+Kingdom..pdf?MOD=AJPERES&CONVERT_TO=url&CACHEID=ROOTWORKSPACE-e5bbede0-1119-4f70-b4ef-f043ce58d780-p6OMj1M',
 'PDPLIR': 'https://dgp.sdaia.gov.sa/wps/wcm/connect/2a9a4744-a2ae-444f-bad2-6dacde843637/The+impelementing+regulation+of+the+personal+data+protection+law.pdf?MOD=AJPERES&CONVERT_TO=url&CACHEID=ROOTWORKSPACE-2a9a4744-a2ae-444f-bad2-6dacde843637-pSK-Oie',
}
for k, u in URLS.items():
    try:
        r = requests.get(u, timeout=25, headers={'User-Agent': 'Mozilla/5.0'})
        print(f"{k}: {r.status_code} · {r.headers.get('Content-Type','?')} · {len(r.content):,} bytes")
    except Exception as e:
        print(f"{k}: FAILED · {type(e).__name__}")

DTR: 200 · application/pdf · 1,676,422 bytes
PDPLIR: 200 · application/pdf · 779,360 bytes


In [11]:
import pathlib
p = pathlib.Path(f'{KB}/app.py'); s = p.read_text(encoding='utf-8')
n = 0
def sub(o, ne):
    global s, n
    if o in s: s = s.replace(o, ne); n += 1
    else: print('MISS:', o[:55])

# 4 — em dash instead of an empty cell
sub('''f"{g['best']:.2f}" if g["best"] else "",''',
    '''f"{g['best']:.2f}" if g["best"] else "\\u2014",''')

# 5 — headers short enough not to wrap
sub('''headers=["subdomain", "title", "accepted", "review",
                             "absence", "untested", "clauses", "reading"],''',
    '''headers=["sub", "title", "acc.", "rev.",
                             "absent", "untest.", "total", "reading"],''')

# 6 — an empty table reads as a broken view; say why it is empty instead
sub('''                if default:
                    app.load(lambda: clause_detail(d, default), None, [info, table])''',
    '''                if default:
                    app.load(lambda: clause_detail(d, default), None, [info, table])
                gr.Markdown(
                    "A clause with no rows above has no accepted mapping: see "
                    "the Gaps tab, which shows whether a candidate was offered "
                    "and rejected, or none reached the agents at all."
                )''')

# 7 — the url column was truncated to 70 chars, so it read as a broken link
sub('''m.get("reference_pair", "") or "\\u2014", m.get("url", "")[:70],''',
    '''m.get("reference_pair", "") or "\\u2014", m.get("url", ""),''')

p.write_text(s, encoding='utf-8'); print(f'✓ {n}/4')

MISS: headers=["subdomain", "title", "accepted", "review",
  
✓ 3/4


In [15]:
!cd {KB} && python debate.py compare

relationship          single judge  after debate   delta
  equal                          0            15     +15
  intersects_with              181           101     -80
  not_related                    0             7      +7
  subset_of                     66           124     +58
  superset_of                   17            17      +0

confidence                  before         after
  mean                       0.559         0.667
  accepted                      73           212    +139

A shift out of intersects_with is the result to look for: it is the
verdict a single judge reaches when it has not worked out which side
contains the other.


In [16]:
!cd {KB} && python debate.py report

DEBATE REPORT
findings debated   : 264
objections raised  : 235 (89%)
mappings changed   : 156 (59%)

verdicts
  revise                159
  uphold                 79
  reject                 14
  (none)                 11
  unresolved              1

grounds of objection
  wrong_type            112
  wrong_control          95
  wrong_direction        19
  not_related             6
  misread_evidence        3

relationship before -> after (changed only)
  3.1.1-4.b          SA-9       intersects_with  -> intersects_with
  3.1.1-10           SA-2       intersects_with  -> subset_of
  3.1.2-1            PM-9       intersects_with  -> subset_of
  3.1.3-1            SC-1       intersects_with  -> superset_of
  3.1.3-1            PS-1       intersects_with  -> subset_of
  3.1.3-1            RA-1       intersects_with  -> subset_of
  3.1.3-2            RA-1       intersects_with  -> intersects_with
  3.1.3-2            PS-1       intersects_with  -> intersects_with
  3.1.3-4.d          PS-9 

In [17]:
import json
from collections import Counter

rows = [json.loads(l) for l in open(f'{KB}/processed/findings_v2.jsonl', encoding='utf-8')]

errored   = [r for r in rows if r.get('error')]
unresolved= [r for r in rows if not r.get('error')
             and (r.get('final', {}).get('status') in (None, 'unresolved'))]
applied   = [r for r in rows if not r.get('error')
             and r.get('final', {}).get('status') not in (None, 'unresolved')]

print(f"صفوف findings_v2.jsonl        : {len(rows)}   ← ما يطبعه compare")
print(f"  منها أحكام طُبِّقت على النتائج : {len(applied)}   ← ما يقوله التقرير")
print(f"  unresolved (امتنع الفاصل)    : {len(unresolved)}")
print(f"  نداءات فشلت                  : {len(errored)}")
print()
print('حالات final:', Counter(r.get('final',{}).get('status') for r in rows))
print('الأحكام    :', Counter(r.get('verdict') for r in rows))

صفوف findings_v2.jsonl        : 264   ← ما يطبعه compare
  منها أحكام طُبِّقت على النتائج : 252   ← ما يقوله التقرير
  unresolved (امتنع الفاصل)    : 1
  نداءات فشلت                  : 11

حالات final: Counter({'accepted': 212, 'disputed': 26, 'rejected_by_debate': 14, 'unresolved': 12})
الأحكام    : Counter({'revise': 159, 'uphold': 79, 'reject': 14, '': 11, 'unresolved': 1})


In [18]:
import json, pathlib
p = pathlib.Path(f'{KB}/processed/findings_v2.jsonl')
rows = [json.loads(l) for l in p.open(encoding='utf-8')]
kept = [r for r in rows if not r.get('error')]
p.write_text('\n'.join(json.dumps(r, ensure_ascii=False) for r in kept) + '\n',
             encoding='utf-8')
print(f'{len(rows)} → {len(kept)}  (أُسقط {len(rows)-len(kept)} للإعادة)')

264 → 253  (أُسقط 11 للإعادة)


In [19]:
import os
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
os.environ['OPENAI_API_KEY']    = userdata.get('OPENAI_API_KEY')
!pip install -q anthropic openai
!cd {KB} && python debate.py run --band provisional

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.6 MB/s eta 0:00:00
resuming: 253 already debated
debating 11 finding(s) from band 'provisional'
  opponent: openai:gpt-4o
  arbiter : anthropic:claude-sonnet-5

  10/11  objections 8  changed 5  eta 0.2 min
  11/11  objections 9  changed 5  eta 0.0 min

objections raised : 9 of 11
verdicts          : uphold=5, revise=5, none=1
verdict changed the mapping: 5

relationship shifts
  superset_of->intersects_with             1
  subset_of->intersects_with               1
  intersects_with->subset_of               1
  intersects_with->intersects_with         1
  intersects_with->superset_of             1

wrote -> processed/findings_v2.jsonl

findings.jsonl is untouched. Compare with `python debate.py compare`.


In [20]:
import json
rows = [json.loads(l) for l in open(f'{KB}/processed/findings_v2.jsonl', encoding='utf-8')]
bad = [r for r in rows if r.get('error')]
print(f'صفوف: {len(rows)}  |  فاشل: {len(bad)}')
for r in bad:
    print(f"  {r['clause_id']} → {r['control_id']}")
    print(f"  {r['error'][:160]}")

صفوف: 264  |  فاشل: 1
  3.4.3-4.a.3 → SA-9
  no JSON object in the response


In [21]:
import json, pathlib
p = pathlib.Path(f'{KB}/processed/findings_v2.jsonl')
rows = [json.loads(l) for l in p.open(encoding='utf-8')]
kept = [r for r in rows if not r.get('error')]
p.write_text('\n'.join(json.dumps(r, ensure_ascii=False) for r in kept) + '\n', encoding='utf-8')
print(f'{len(rows)} → {len(kept)}')

264 → 263


In [22]:
!cd {KB} && python debate.py run --band provisional

resuming: 263 already debated
debating 1 finding(s) from band 'provisional'
  opponent: openai:gpt-4o
  arbiter : anthropic:claude-sonnet-5

  1/1  objections 1  changed 1  eta 0.0 min

objections raised : 1 of 1
verdicts          : revise=1
verdict changed the mapping: 1

relationship shifts
  intersects_with->subset_of               1

wrote -> processed/findings_v2.jsonl

findings.jsonl is untouched. Compare with `python debate.py compare`.


In [23]:
!cd {KB} && python app.py selftest 2>&1 | head -10

data ready: True 
  clauses 489  controls 1196 (1014 active)  findings rows 407
  accepted 335  verified 512
  arbiter rulings applied: 263
  clause gaps 119: not tested=53, below threshold=49, evidenced absence=17
  local specificity (superset_of): 20
  excluded as list items / referential: 126 of 407
  weakest subdomain: 3.3.8 — 4 no match, 2 for review, of 11  [evidenced absence]
  mapping opens on: 3.3.8-6.g
  control gaps: worst family CP 100%


In [24]:
import json
from collections import Counter
import sys; sys.path.insert(0, KB)
import app as A
d = A.Data()
print(Counter(f['status'] for r in d.findings for f in r['findings']))

Counter({'accepted': 335, 'disputed': 177, 'rejected': 29})


In [25]:
from collections import Counter
import sys; sys.path.insert(0, KB)
import app as A
d = A.Data()

ok = lambda cid: (not A.enumerative(d.clauses.get(cid,{}))
                  and d.clauses.get(cid,{}).get('clause_type') != 'referential')
m   = sum(1 for r in d.findings if ok(r['clause_id']))
acc = len({cid for cid,_ in d.accepted if ok(cid)})
ver = len({cid for cid,_ in d.verified if ok(cid)})

print(f"مقبول : {acc}/{m} = {100*acc/m:.1f}%   (كان 161 / 57.3%)")
print(f"متحقَّق: {ver}/{m} = {100*ver/m:.1f}%   (كان 205 / 73.0%)")
print(Counter(f['relationship'] for _, f in d.verified))
hard, soft = A.parameter_gaps(d)
print(f"معامل: {len(hard)} صلبة · {len(soft)} ليّنة   (كان 6 / 62)")

مقبول : 162/281 = 57.7%   (كان 161 / 57.3%)
متحقَّق: 205/281 = 73.0%   (كان 205 / 73.0%)
Counter({'intersects_with': 236, 'subset_of': 221, 'equal': 35, 'superset_of': 20})
معامل: 6 صلبة · 64 ليّنة   (كان 6 / 62)


In [26]:
!cd {KB} && python app.py selftest

data ready: True 
  clauses 489  controls 1196 (1014 active)  findings rows 407
  accepted 335  verified 512
  arbiter rulings applied: 263
  clause gaps 119: not tested=53, below threshold=49, evidenced absence=17
  local specificity (superset_of): 20
  excluded as list items / referential: 126 of 407
  weakest subdomain: 3.3.8 — 4 no match, 2 for review, of 11  [evidenced absence]
  mapping opens on: 3.3.8-6.g
  control gaps: worst family CP 100%
  parameter gaps: hard 6 soft 64
  drift: {'SR': {'total': 27, 'matched': 2, 'title': 'Supply Chain Risk Management'}, 'PT': {'total': 21, 'matched': 1, 'title': 'Personally Identifiable Information Processing and Transparency'}, 'rev520': {'known': ['SA-15(13)', 'SA-24', 'SI-2(7)'], 'matched': ['SI-2(7)']}}
  clause choices 404
  detail rows for 3.1.1-1: 0
  kb rows 12
  itu factors 6 dimensions 13

## SANAD — evidence-anchored regulatory alignment

**SAMA Cyber Security Framework (2017)** against **NIST SP 800-53 Rev 5.2.0**,
with every cl

In [27]:
import json
from collections import Counter
cl = [json.loads(l) for l in open(f'{KB}/processed/sama_clauses.jsonl', encoding='utf-8')]
print(f'sama_clauses.jsonl : {len(cl)} صفاً')
print(Counter(c['clause_type'] for c in cl))
print('طرفية:', sum(1 for c in cl if not c['n_children']))
print('جذور :', sum(1 for c in cl if c['n_children']))

fin = [json.loads(l) for l in open(f'{KB}/processed/findings.jsonl', encoding='utf-8')]
print(f'\nfindings.jsonl     : {len(fin)} بنداً محكوماً')

ids_cl, ids_fin = {c['clause_id'] for c in cl}, {r['clause_id'] for r in fin}
print('محكوم وغير موجود في المحلّل:', sorted(ids_fin - ids_cl)[:5] or 'لا شيء')

sama_clauses.jsonl : 493 صفاً
Counter({'substantive': 409, 'stem': 79, 'referential': 5})
طرفية: 412
جذور : 81

findings.jsonl     : 407 بنداً محكوماً
محكوم وغير موجود في المحلّل: لا شيء


In [28]:
import json
from collections import Counter
cl = [json.loads(l) for l in open(f'{KB}/processed/sama_clauses.jsonl', encoding='utf-8')]
dup = {k: v for k, v in Counter(c['clause_id'] for c in cl).items() if v > 1}
print('معرّفات مكرّرة:', dup)
for cid in dup:
    print(f'\n--- {cid}')
    for c in cl:
        if c['clause_id'] == cid:
            print(f"   ص{c['page']}  [{c['clause_type']}]  {c['text'][:80]}")

معرّفات مكرّرة: {'3.3.13-1': 2, '3.3.13-2': 2, '3.3.13-3': 2, '3.3.13-4': 2}

--- 3.3.13-1
   ص31  [substantive]  The cyber security standards for electronic banking services should be defined, 
   ص32  [substantive]  sign-on;

--- 3.3.13-2
   ص31  [substantive]  The compliance with cyber security standards for electronic banking services sho
   ص32  [substantive]  adding or modifying beneficiaries;

--- 3.3.13-3
   ص31  [substantive]  The effectiveness of the cyber security standard for electronic banking services
   ص32  [substantive]  adding utility and government payment services;

--- 3.3.13-4
   ص31  [stem]  Electronic banking services security standard should cover:
   ص32  [stem]  high-risk transactions (when it exceeds predefined limits);


In [29]:
import json
fin = [json.loads(l) for l in open(f'{KB}/processed/findings.jsonl', encoding='utf-8')]
for cid in ['3.3.13-1','3.3.13-2','3.3.13-3','3.3.13-4']:
    r = [x for x in fin if x['clause_id'] == cid]
    print(f"{cid}: {len(r)} صفاً | نصّه: {r[0]['clause_text'][:60] if r else '—'}")

3.3.13-1: 2 صفاً | نصّه: sign-on;
3.3.13-2: 2 صفاً | نصّه: adding or modifying beneficiaries;
3.3.13-3: 2 صفاً | نصّه: adding utility and government payment services;
3.3.13-4: 0 صفاً | نصّه: —


In [30]:
import pathlib
p = pathlib.Path(f'{KB}/appendix.py'); s = p.read_text(encoding='utf-8')
old = 'debate layer; the debated run accepts 328 judgments at the same threshold,'
new = 'debate layer; the debated run accepts 335 judgments at the same threshold,'
assert old in s, 'لم يُعثر'
p.write_text(s.replace(old, new), encoding='utf-8')
print('✓ 328 → 335')

✓ 328 → 335


In [31]:
!cd {KB} && python appendix.py > appendix.tex
import re
s = open(f'{KB}/appendix.tex', encoding='utf-8').read()
print('عربية      :', bool(re.search(r'[\u0600-\u06FF]', s)))
print('أرقام قديمة:', [x for x in ['328','252','184','161','57.3','255','111','120 cov']
                       if x in s] or 'لا شيء')
print('الملاحق    :', re.findall(r'\\section\*\{([^}]+)\}', s))
print('صفوف الجدول:', s.count('\\texttt{SAU-') + s.count('\\texttt{USA-') + s.count('\\texttt{INT-'))

عربية      : True
أرقام قديمة: ['184', '111']
الملاحق    : ['Appendix A — References', 'Appendix B — ITU AI Readiness factors and dimensions', 'Appendix C — Knowledge base manifest', 'Appendix D — Evaluation detail']
صفوف الجدول: 15


In [32]:
!cd {KB} && python appendix.py > appendix.tex

import re
s = open(f'{KB}/appendix.tex', encoding='utf-8').read()
print('عربية      :', bool(re.search(r'[\u0600-\u06FF]', s)))
print('أرقام قديمة:', [x for x in ['328','252','184','161','57.3','255','111'] if x in s] or 'لا شيء')
print('الملاحق    :', re.findall(r'\\section\*\{([^}]+)\}', s))
print('صفوف C     :', s.count('\\texttt{SAU-') + s.count('\\texttt{USA-') + s.count('\\texttt{INT-'))
print()
print('--- D.3 ---')
m = re.search(r'D\.3 Risk.*?end\{tabular\}', s, re.S)
print(m.group()[:700] if m else 'لم يوجد')

عربية      : True
أرقام قديمة: ['184', '111']
الملاحق    : ['Appendix A — References', 'Appendix B — ITU AI Readiness factors and dimensions', 'Appendix C — Knowledge base manifest', 'Appendix D — Evaluation detail']
صفوف C     : 15

--- D.3 ---
D.3 Risk--coverage}
\noindent Computed on the single-judge findings, before the
debate layer; the debated run accepts 335 judgments at the same threshold,
because arbiter rulings resolve cases the proposer left under-confident.
\noindent Confidence is not continuous: it clusters, so the
threshold selects a band rather than a point.
\begin{center}\small
\begin{tabular}{@{}rrr@{}}\toprule
\hd{$\tau$} & \hd{Accepted judgments} & \hd{Coverage} \\
\midrule
0.40 & 491 & 93\% \\
0.50 & 377 & 71\% \\
0.55 & 318 & 60\% \\
\textbf{0.60} & \textbf{186} & \textbf{35\%} \\
0.65 & 146 & 27\% \\
0.70 & 113 & 21\% \\
0.80 & 46 & 8\% \\
\bottomrule\end{tabular}


In [33]:
import pathlib, re

p = pathlib.Path(f'{KB}/appendix.py'); s = p.read_text(encoding='utf-8')
done = []

# ١ — حذف العناوين العربية (لا يعرضها pdflatex)
old = '''    if r.get("title_ar"):
        bits.append(f"({tex(r['title_ar'])}).")
'''
if old in s:
    s = s.replace(old, ''); done.append('حُذفت العربية')

# ٢ — فقرتا D.3 تندمجان: \noindent مكرّر بلا فاصل فقرة
old2 = '''because arbiter rulings resolve cases the proposer left under-confident.""")
        print(r"""\\noindent Confidence is not continuous: it clusters, so the
threshold selects a band rather than a point.""")'''
new2 = '''because arbiter rulings resolve cases the proposer left under-confident.
Confidence is not continuous: it clusters, so the threshold selects a band
rather than a point.""")'''
if old2 in s:
    s = s.replace(old2, new2); done.append('دُمجت فقرتا D.3')

p.write_text(s, encoding='utf-8')
print('تم:', done or 'لا شيء — مُطبَّق سابقاً')

تم: ['حُذفت العربية', 'دُمجت فقرتا D.3']


In [34]:
!cd {KB} && python appendix.py > appendix.tex

import re
s = open(f'{KB}/appendix.tex', encoding='utf-8').read()
print('عربية:', bool(re.search(r'[\u0600-\u06FF]', s)))
print()
for num in ['184', '111']:
    for m in re.finditer(num, s):
        print(f'  {num} في: ...{s[max(0,m.start()-45):m.start()+25]}...')

عربية: False

  184 في: ... & Risk and trust & -- & acceptance gate and 184 declared referrals (\...
  111 في: ...://dgp.sdaia.gov.sa/wps/wcm/connect/e5bbede0-1119-4f70-b4ef-f043ce58d7...
  111 في: ...ONVERT_TO=url&CACHEID=ROOTWORKSPACE-e5bbede0-1119-4f70-b4ef-f043ce58d7...


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
KB = '/content/drive/MyDrive/sanad-ai-readiness/kb'
os.environ['HF_HOME'] = '/content/drive/MyDrive/sanad-ai-readiness/.hf'

!pip install -q "gradio==5.9.1"
import gradio; print('gradio', gradio.__version__)

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.4/320.4 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 131.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.2/170.2 kB 18.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
python-fasthtml 0.14.6 requires starlette>=1.0.1, but you have starlette 0.52.1 which is incompatible.
google-adk 2.4.0 requires starlette<2,>=1.0.1, but you have starlette 0.52.1 which is incompatible.
google-adk 2.4.0 requires websockets<16,>=15.0.1, but you have websockets 14.2 which is incompatible.
langsmith 0.10.2 requires websockets>=15.0, but you have websockets 14.2 which is incompatible.

open(f'{KB}/../requirements.txt','w').write("""gradio==5.9.1
sentence-transformers
numpy
pdfplumber
pypdf
requests
beautifulsoup4
lxml
anthropic
openai
""")
print(open(f'{KB}/../requirements.txt').read())

In [3]:
import re, pathlib
s = pathlib.Path(f'{KB}/app.py').read_text(encoding='utf-8')
m = re.search(r'headers=\["sub.*?\],\s*\n\s*column_widths=\[[^\]]*\],', s, re.S)
print(m.group() if m else 'لم يوجد نمط sub')

if not m:
    for mm in re.finditer(r'headers=\[[^\]]*\]', s):
        print(repr(mm.group()[:130]))

لم يوجد نمط sub
'headers=["clause", "subdomain", "severity", "best confidence",\n                             "controls offered and not taken", "tex'
'headers=["subdomain", "title", "accepted", "for review",\n                             "evidenced absence", "not tested", "clauses"'
'headers=["family", "title", "controls",\n                             "no accepted match", "reached by nothing at all",\n           '
'headers=["clause", "subdomain", "control", "confidence",\n                             "status", "clause text"]'
'headers=["clause", "control", "SAMA value", "note", "clause text"]'
'headers=["factor", "documents", "note"]'
'headers=["dimension", "documents"]'


In [4]:
import pathlib
p = pathlib.Path(f'{KB}/app.py'); s = p.read_text(encoding='utf-8')
n = 0
def sub(o, ne):
    global s, n
    if o in s: s = s.replace(o, ne); n += 1
    else: print('MISS:', o[:60])

# جدول الفجوات
sub('''headers=["clause", "subdomain", "severity", "best confidence",
                             "controls offered and not taken", "text"],''',
    '''headers=["clause", "subdomain", "severity", "conf",
                             "offered, not taken", "clause text"],
                    column_widths=["11%", "19%", "13%", "6%", "19%", "32%"],''')

# جدول النطاقات الفرعية
sub('''headers=["subdomain", "title", "accepted", "for review",
                             "evidenced absence", "not tested", "clauses",
                             "reading"],''',
    '''headers=["sub", "title", "acc", "rev", "gap", "untd", "all", "reading"],
                    column_widths=["9%", "24%", "6%", "6%", "6%", "7%", "6%", "36%"],''')

# جدول العائلات
sub('''headers=["family", "title", "controls",
                             "no accepted match", "reached by nothing at all",
                             "documented exclusion"],''',
    '''headers=["fam", "title", "n", "no acc.", "unreached", "note"],
                    column_widths=["6%", "26%", "6%", "11%", "12%", "39%"],''')

# الشرطة بدل الفراغ
sub('''f"{g['best']:.2f}" if g["best"] else "",''',
    '''f"{g['best']:.2f}" if g["best"] else "\\u2014",''')

p.write_text(s, encoding='utf-8'); print(f'✓ {n}/4')

MISS: f"{g['best']:.2f}" if g["best"] else "",
✓ 3/4


In [5]:
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(7860)"))

https://7860-gpu-t4-s-kkb-usw1b2-360bhvhuizmtl-b.us-west1-2.prod.colab.dev


In [24]:
!cd {KB} && python app.py

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://99e08251da33aa7eb4.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Keyboard interruption in main thread... closing server.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2869, in block_thread
    time.sleep(0.1)
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/drive/MyDrive/sanad-ai-readiness/kb/app.py", line 887, in <module>
    sys.exit(selftest() if "selftest" in sys.argv else launch("--no-share" not in sys.argv))
                                                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/sanad-ai-readiness/kb/app.py", line 880, in launch

In [7]:
import pathlib, re
p = pathlib.Path(f'{KB}/app.py'); s = p.read_text(encoding='utf-8')

before = s.count('wrap=True')
s = s.replace('wrap=True,', 'wrap=False,').replace('wrap=True)', 'wrap=False)')
p.write_text(s, encoding='utf-8')
print(f'✓ {before} جدولاً: wrap=True → False')

✓ 8 جدولاً: wrap=True → False


In [9]:
import pathlib
p = pathlib.Path(f'{KB}/app.py'); s = p.read_text(encoding='utf-8')
n = s.count('wrap=False')
s = s.replace('wrap=False,', 'wrap=True,').replace('wrap=False)', 'wrap=True)')
p.write_text(s, encoding='utf-8')
print(f'✓ أُعيد {n} جدولاً إلى wrap=True')

✓ أُعيد 8 جدولاً إلى wrap=True


In [11]:
import pathlib
p = pathlib.Path(f'{KB}/app.py'); s = p.read_text(encoding='utf-8')
n = 0
def sub(o, ne):
    global s, n
    if o in s: s = s.replace(o, ne); n += 1
    else: print('MISS:', o[:55])

sub('"offered, not taken", "clause text"],',
    '"shortlisted", "clause text"],')

sub('''                    "tested** means no plausible candidate ever reached it — a "''',
    '''                    "tested** means the shortlist never reached a control family "
                    "the subdomain plausibly belongs to — a "''')

p.write_text(s, encoding='utf-8'); print(f'✓ {n}/2')

✓ 2/2


In [13]:
import sys; sys.path.insert(0, KB)
import importlib, app as A; importlib.reload(A)
from collections import Counter
d = A.Data()

mapped = set(A.SUBDOMAIN_EXPECTED)
all_sd = {c['subdomain'] for c in d.clauses.values() if c.get('subdomain')}
missing = sorted(all_sd - mapped, key=lambda s: [int(x) for x in s.split('.')])
print(f'نطاقات بلا إدخال: {len(missing)} من {len(all_sd)}')
print(missing)

g = A.clause_gaps(d)
untested = [x for x in g if x['severity'] == 'not tested']
forced = [x for x in untested if x['sd_key'] in missing]
print(f'\nnot tested: {len(untested)}  منها {len(forced)} بسبب نطاق غير مُدرَج')
print(Counter(x['sd_key'] for x in forced).most_common(8))

نطاقات بلا إدخال: 15 من 36
['3.1.1', '3.1.2', '3.1.3', '3.1.4', '3.2.1.1', '3.2.1.2', '3.2.1.3', '3.2.1.4', '3.2.2', '3.2.3', '3.2.4', '3.3.4', '3.3.10', '3.3.12', '3.3.13']

not tested: 53  منها 53 بسبب نطاق غير مُدرَج
[('3.3.13', 16), ('3.1.3', 9), ('3.1.4', 8), ('3.2.1.3', 8), ('3.1.1', 6), ('3.1.2', 3), ('3.2.1.4', 1), ('3.2.2', 1)]


In [14]:
import pathlib
p = pathlib.Path(f'{KB}/app.py'); s = p.read_text(encoding='utf-8')

ADD = '''    # governance and risk subdomains, completing the map to all 36
    "3.1.1": {"PM", "PL"},          # cyber security governance
    "3.1.2": {"PM", "PL"},          # cyber security strategy
    "3.1.3": {"PM", "PL", "SI"},    # cyber security policy
    "3.1.4": {"PM", "PS", "PL"},    # roles and responsibilities
    "3.2.1.1": {"RA", "PM"},        # risk identification
    "3.2.1.2": {"RA"},              # risk analysis
    "3.2.1.3": {"RA", "PM"},        # risk response
    "3.2.1.4": {"RA", "CA"},        # risk monitoring
    "3.2.2": {"CA", "PM"},          # regulatory compliance
    "3.2.3": {"CA", "PM"},          # compliance with industry standards
    "3.2.4": {"CA", "RA"},          # review and audit
    "3.3.4": {"SC", "CM"},          # cryptography / architecture
    "3.3.10": {"SC", "MP"},         # backup and recovery
    "3.3.12": {"SC", "SI"},         # payment systems (referential)
    "3.3.13": {"AC", "IA", "SC"},   # electronic banking services
}'''

old = '''    "3.4.1": {"SA", "SR"}, "3.4.2": {"SA", "SR"}, "3.4.3": {"SA", "SC"},
}'''
assert old in s, 'لم يُعثر'
p.write_text(s.replace(old, '''    "3.4.1": {"SA", "SR"}, "3.4.2": {"SA", "SR"}, "3.4.3": {"SA", "SC"},
''' + ADD), encoding='utf-8')
print('✓ اكتملت الستة والثلاثون')

✓ اكتملت الستة والثلاثون


In [15]:
import pathlib
p = pathlib.Path(f'{KB}/app.py'); s = p.read_text(encoding='utf-8')

old = '''"**Below threshold** is a match, not a gap.\\n\\n"'''
new = '''"**Below threshold** is a match, not a gap. The "
                    "evidenced-absence / not-tested split depends on a "
                    "hand-curated family map covering the routed subdomains; "
                    "broadening that map would reclassify some of the 53, not "
                    "change any judgment.\\n\\n"'''

assert old in s, 'لم يُعثر — أرسلي السطر الحالي'
p.write_text(s.replace(old, new), encoding='utf-8')
print('✓ أُدرجت جملة الإفصاح')

✓ أُدرجت جملة الإفصاح


In [25]:
!cd {KB} && python app.py selftest

data ready: True 
  clauses 489  controls 1196 (1014 active)  findings rows 407
  accepted 335  verified 512
  arbiter rulings applied: 263
  clause gaps 119: not tested=53, below threshold=49, evidenced absence=17
  local specificity (superset_of): 20
  excluded as list items / referential: 126 of 407
  weakest subdomain: 3.3.8 — 4 no match, 2 for review, of 11  [evidenced absence]
  mapping opens on: 3.3.8-6.g
  control gaps: worst family CP 100%
  parameter gaps: hard 6 soft 64
  drift: {'SR': {'total': 27, 'matched': 2, 'title': 'Supply Chain Risk Management'}, 'PT': {'total': 21, 'matched': 1, 'title': 'Personally Identifiable Information Processing and Transparency'}, 'rev520': {'known': ['SA-24', 'SA-15(13)', 'SI-2(7)'], 'matched': ['SI-2(7)']}}
  clause choices 404
  detail rows for 3.1.1-1: 0
  kb rows 12
  itu factors 6 dimensions 13

## SANAD — evidence-anchored regulatory alignment

**SAMA Cyber Security Framework (2017)** against **NIST SP 800-53 Rev 5.2.0**,
with every cl

In [26]:
!cd {KB} && python validate_manifest.py manifest.csv

OK — 15 rows, tiers {'core': 8, 'method': 3, 'referenced': 4}


In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
KB = '/content/drive/MyDrive/sanad-ai-readiness/kb'
os.environ['HF_HOME'] = '/content/drive/MyDrive/sanad-ai-readiness/.hf'
!ls {KB}/processed/review*.jsonl

Mounted at /content/drive
/content/drive/MyDrive/sanad-ai-readiness/kb/processed/review_debated.jsonl
/content/drive/MyDrive/sanad-ai-readiness/kb/processed/review.jsonl


In [2]:
!cd {KB} && python precision.py score

PRECISION OF ACCEPTED MAPPINGS
labelled            : 30
excluded as unsure  : 0
decided             : 30

correct             : 27
wrong relationship  : 0
not a relationship  : 3

precision           : 90.0%   95% CI [74.4%, 96.5%]

by confidence band
  confirmed     13/14   92.9%  [69%, 99%]
  provisional   14/16   87.5%  [64%, 97%]

by relationship claimed
  equal                1/1
  intersects_with      8/8
  subset_of           18/21

the 3 that failed
  3.1.3-1            RA-1       subset_of        wrong
  3.3.8-6.g          SI-2       subset_of        wrong
  3.1.3-1            PS-1       subset_of        wrong

Report as: precision 90% on a stratified sample of 30 accepted mappings, 95% CI [74%, 97%].
A sample this size supports a range, not a point estimate — quote the
interval, and state the sample size beside the figure.


In [17]:
import sys; sys.path.insert(0, KB)
import importlib, app as A; importlib.reload(A)
print('عدد النطاقات في الخريطة:', len(A.SUBDOMAIN_EXPECTED))
print('3.2.1.3 موجود؟', '3.2.1.3' in A.SUBDOMAIN_EXPECTED)

عدد النطاقات في الخريطة: 36
3.2.1.3 موجود؟ True


In [19]:
import re, pathlib
s = pathlib.Path(f'{KB}/app.py').read_text(encoding='utf-8')
m = re.search(r'SUBDOMAIN_EXPECTED: Dict\[str, set\] = \{.*?\n\}', s, re.S)
print(m.group() if m else 'لم يوجد')

SUBDOMAIN_EXPECTED: Dict[str, set] = {
    "3.1.5": {"SA", "PL"}, "3.1.6": {"AT"}, "3.1.7": {"AT"},
    "3.2.1": {"RA", "PM"}, "3.2.5": {"CA", "AU"}, "3.3.1": {"PS"},
    "3.3.2": {"PE"}, "3.3.3": {"CM"}, "3.3.5": {"AC", "IA"},
    "3.3.6": {"SA", "SI"}, "3.3.7": {"CM"}, "3.3.8": {"SC", "CM"},
    "3.3.9": {"SC"}, "3.3.11": {"MP"}, "3.3.14": {"AU", "SI"},
    "3.3.15": {"IR"}, "3.3.16": {"RA", "SI"}, "3.3.17": {"RA", "SI"},
    "3.4.1": {"SA", "SR"}, "3.4.2": {"SA", "SR"}, "3.4.3": {"SA", "SC"},
    # governance and risk subdomains, completing the map to all 36
    "3.1.1": {"PM", "PL"},          # cyber security governance
    "3.1.2": {"PM", "PL"},          # cyber security strategy
    "3.1.3": {"PM", "PL", "SI"},    # cyber security policy
    "3.1.4": {"PM", "PS", "PL"},    # roles and responsibilities
    "3.2.1.1": {"RA", "PM"},        # risk identification
    "3.2.1.2": {"RA"},              # risk analysis
    "3.2.1.3": {"RA", "PM"},        # risk response
    "3.2.1.4": {"RA

In [21]:
import pathlib
lines = pathlib.Path(f'{KB}/app.py').read_text(encoding='utf-8').split('\n')
for i in range(162, 182):
    print(f'{i+1:>4}: {lines[i]}')

 163: # A clause whose shortlist contained nothing from a family plausibly related to
 164: # its subdomain was never really tested. Calling that a gap asserts absence from
 165: # the catalogue on the strength of a retrieval failure.
 166: SUBDOMAIN_EXPECTED: Dict[str, set] = {
 167:     "3.1.5": {"SA", "PL"}, "3.1.6": {"AT"}, "3.1.7": {"AT"},
 168:     "3.2.1": {"RA", "PM"}, "3.2.5": {"CA", "AU"}, "3.3.1": {"PS"},
 169:     "3.3.2": {"PE"}, "3.3.3": {"CM"}, "3.3.5": {"AC", "IA"},
 170:     "3.3.6": {"SA", "SI"}, "3.3.7": {"CM"}, "3.3.8": {"SC", "CM"},
 171:     "3.3.9": {"SC"}, "3.3.11": {"MP"}, "3.3.14": {"AU", "SI"},
 172:     "3.3.15": {"IR"}, "3.3.16": {"RA", "SI"}, "3.3.17": {"RA", "SI"},
 173:     "3.4.1": {"SA", "SR"}, "3.4.2": {"SA", "SR"}, "3.4.3": {"SA", "SC"},
 174: },   # electronic banking services
 175: }
 176: 
 177: 
 178: def enumerative(c: Dict[str, Any]) -> bool:
 179:     """A list item, not a requirement.
 180: 
 181:     "cyber security specialists;" carries no 

In [22]:
import pathlib
p = pathlib.Path(f'{KB}/app.py')
lines = p.read_text(encoding='utf-8').split('\n')

assert lines[173].strip().startswith('},'), f'غير متوقّع: {lines[173]!r}'
assert lines[174].strip() == '}',            f'غير متوقّع: {lines[174]!r}'

lines[173:175] = ['}']
p.write_text('\n'.join(lines), encoding='utf-8')

import importlib, sys
sys.path.insert(0, KB)
import app as A; importlib.reload(A)
print('✓ النطاقات:', len(A.SUBDOMAIN_EXPECTED), '| 3.2.1.3:', '3.2.1.3' in A.SUBDOMAIN_EXPECTED)

✓ النطاقات: 21 | 3.2.1.3: False


In [23]:
!cd {KB} && python app.py selftest 2>&1 | grep -E "clause gaps|weakest|accepted"

  accepted 335  verified 512
  clause gaps 119: not tested=53, below threshold=49, evidenced absence=17
  weakest subdomain: 3.3.8 — 4 no match, 2 for review, of 11  [evidenced absence]
| judgments | **541** — 335 accepted, 177 for review, 29 rejected at the gate |
